# QuantumJam 2026 · Curva de energía de $\mathrm{H_4}$ con VQE

Cadena lineal de 4 H con espaciado uniforme $d$, base STO-3G, neutra y singlete: 4 orbitales espaciales → 8 espín-orbitales → **8 qubits** (Jordan-Wigner), 4 electrones. Calculamos con VQE la curva $E(d)$ para $d \in [0{,}6;\ 3{,}0]$ Å, con $R_e$, $D_e = E(3{,}0\ \text{Å}) - E(R_e)$ y una barra de error $\pm\sigma$ por punto, y la comparamos con CCSD y con la energía exacta (FCI). Enunciado completo en `docs/challenge/`.

### El problema: por qué la zona estirada es difícil

Cerca del equilibrio ($R_e \approx 0{,}89$ Å) el estado fundamental es casi un solo determinante (Hartree-Fock pesa 0,985 a 0,6 Å) y CCSD es prácticamente exacto (< 0,01 mHa de error). Al estirar la cadena, los 4 electrones terminan en cuatro orbitales 1s casi desacoplados, un singlete con muchos determinantes de peso parecido: el estado se vuelve **multirreferencia** (HF pesa 0,58 a 1,8 Å y 0,27 a 3,0 Å). Un método de una sola referencia como CCSD no lo describe bien, y como no es variacional, **cae por debajo de FCI**:

| $d$ (Å) | 0,9 | 1,6 | **1,8** | 2,0 | 2,4 | 2,6 | 3,0 |
|---|---|---|---|---|---|---|---|
| $E_\text{CCSD} - E_\text{FCI}$ (mHa) | +0,01 | −3,0 | **−9,1** | −18,3 | −30,6 | −30,6 | −23,0 |

(`docs/analysis/data/h4_reference.json`; CCSD(T) es peor todavía). Por eso el desafío pide, para $d \geq 1{,}8$ Å, $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$; 1,8 Å es el punto más exigente porque ahí el margen es el más chico (9,1 mHa). La meta máxima es precisión química, $< 1{,}6$ mHa. CCSD también subestima $D_e$ en ~23 mHa (≈ 290 contra 313 mHa de FCI). Un detalle más: desde ~2,4 Å el primer excitado es un triplete a pocos mHa (0,46 mHa a 3,0 Å), así que la energía sola no alcanza para saber si el estado es el correcto.

### Qué hacemos

| Paso | Sección | Qué sale |
|---|---|---|
| Referencias clásicas HF, CCSD, FCI (solo para comparar y elegir puntos de partida) | 1 | tabla `refs` |
| Hamiltoniano de 8 qubits por Jordan-Wigner, validado contra HF y FCI | 2 | `qubit_hamiltonian(d)` |
| Ansatz **2-QUpCCGSD podado**: 19 excitaciones de qubit como rotaciones de Givens, 465 `cz`, conserva $N$ y $S_z$ | 3 | `build_ansatz()` |
| Búsqueda de parámetros con statevector (permitido), *warm start* entre distancias vecinas y varios arranques | 4 | `params` |
| Energía **reportada**: medición con `SamplerV2` de `qiskit_aer`, shots explícitos y $\sigma$ desde los bitstrings | 4 | tabla `vqe` |
| Curva, $R_e$, $D_e$ y tabla por punto (energía, $\sigma$, shots, `cz`) | 5 | gráficos y tablas |
| Confiabilidad en la zona estirada **sin mirar FCI** ($\langle S^2\rangle$, varianza, robustez) y recién después la comparación con CCSD y FCI | 6 | evidencia y veredicto |
| Resumen de resultados, decisiones, qué no funcionó y limitaciones | Resultados clave, 7 | |

Reglas que respetamos en todo el notebook: el algoritmo es VQE; **ninguna energía reportada sale del statevector** (el statevector solo se usa para buscar parámetros y validar el ansatz); HF, CCSD y FCI son referencias, nunca resultados.

### Cómo leer y correr el notebook

- **Colab:** abrir el notebook y *Entorno de ejecución → Ejecutar todas*. La primera celda instala las dependencias con las versiones fijadas (las mismas que `requirements.txt`). Si Colab pide reiniciar el entorno después del `%pip`, reiniciar y volver a ejecutar todo. Es autocontenido: no importa nada del repo.
- **Local:** `nix develop -c jupyter lab` (mismas versiones; ver `README` del repo).
- La celda **0.1** es el contrato que comparten todas las secciones: grilla `D`, geometría, convención de qubits (`HF_BITSTRING = "00110011"`: qubit $p$ = $(p,\alpha)$, qubit $4+p$ = $(p,\beta)$), transpilación y conteo de `cz` (base `cz, sx, x, rz`, qubits en línea) y los nombres de las tablas `refs`, `params`, `vqe`.
- Cada sección termina con un markdown de **decisiones**: qué elegimos, por qué y qué probamos que no funcionó. La sección 7 los resume y analiza las limitaciones.
- Todas las energías son **totales** (incluyen la repulsión nuclear), en Hartree; los errores, en mHa.
- **Resultados clave** ($R_e$, $D_e$, ¿le ganamos a CCSD?, ¿precisión química?, `cz` y shots) en la celda *Resultados clave*, justo antes de la sección 7.

## 0. Instalación

In [ ]:
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pyscf==2.14.0 numpy scipy pandas matplotlib

## 0.1 Contrato compartido

Nombres, firmas y convenciones que usa todo el notebook. Cada sección reemplaza su `NotImplementedError`; cambios a este contrato se avisan en el issue.

In [ ]:
# Contrato compartido: nombres, firmas y convenciones que usa todo el notebook.
# Cada sección implementa su parte; nadie cambia esto sin avisar en el issue correspondiente.
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap

# --- Grilla y geometría -------------------------------------------------------
BASIS = "sto-3g"
D = np.round(np.concatenate([np.arange(0.6, 1.75, 0.1), np.arange(1.8, 3.01, 0.2)]), 2)  # Å, 19 puntos
STRETCHED = 1.8    # Å: desde acá hay que ganarle a CCSD
DISSOCIATED = 3.0  # Å: D_e = E(3,0) - E(R_e)
CHEM_ACC = 1.6e-3  # Ha


def h4_geometry(d: float) -> str:
    """Cadena lineal de 4 H sobre el eje z, separados por d Å (formato PySCF)."""
    return "; ".join(f"H 0 0 {i * d:.6f}" for i in range(4))


# --- Convención de qubits (Jordan-Wigner) --------------------------------------
# Orbitales espaciales MO de RHF, p = 0..3 en orden de energía.
# qubit p     = espín-orbital (p, α)
# qubit 4 + p = espín-orbital (p, β)
# |1> = ocupado. Qiskit es little-endian: el label de un estado se lee q7 ... q0.
N_ORB, N_QUBITS, N_ALPHA, N_BETA = 4, 8, 2, 2
HF_BITSTRING = "00110011"  # q0, q1 (α 0,1) y q4, q5 (β 0,1) ocupados


def alpha_qubit(p: int) -> int:
    return p


def beta_qubit(p: int) -> int:
    return N_ORB + p


# --- Transpilación y conteo de cz (igual para todos) ---------------------------
BASIS_GATES = ["cz", "sx", "x", "rz"]


def transpile_line(qc: QuantumCircuit) -> QuantumCircuit:
    return transpile(qc, basis_gates=BASIS_GATES, coupling_map=CouplingMap.from_line(qc.num_qubits),
                     optimization_level=3, seed_transpiler=0)


def cz_count(qc: QuantumCircuit) -> int:
    return transpile_line(qc).count_ops().get("cz", 0)


# --- Firmas que implementa cada sección ----------------------------------------
# Tablas compartidas (energías totales en Ha, con repulsión nuclear):
#   refs   : DataFrame  d, e_hf, e_ccsd, e_fci           (#1)
#   params : dict       d -> np.ndarray θ*               (#5)
#   vqe    : DataFrame  d, energy, sigma, shots, cz      (#6)

def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'}."""
    raise NotImplementedError


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    """#2. (H electrónico como SparsePauliOp de 8 qubits en la convención de arriba, e_nuc)."""
    raise NotImplementedError


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito parametrizado de 8 qubits que arranca en HF_BITSTRING."""
    raise NotImplementedError


## 1. Referencias clásicas (HF, CCSD, FCI)

Energías de referencia en toda la grilla `D` con PySCF. **FCI** es la respuesta exacta en la base STO-3G, contra la que se mide todo. **CCSD** es el método clásico a superar en la zona estirada. Ninguna de las dos es un resultado del VQE: solo sirven para comparar y para elegir puntos de partida.

Esta celda define `references(d)` y la tabla `refs` (`d, e_hf, e_ccsd, e_fci`), que también se guarda en `results/classical.csv`. Usa el mismo procedimiento de RHF que `h4_scf` de la sección 2 (`conv_tol = 1e-11`). Las energías no dependen del signo de los orbitales. Para cualquier cosa que dependa de los orbitales (vector CI, amplitudes), usar `h4_scf(d)`.

**Decisiones**

- Se usa **RHF**, no UHF. Desde ~1,2 Å RHF tiene una inestabilidad RHF → UHF: la solución UHF baja la energía pero rompe la simetría de espín. Queremos un singlete con orbitales restringidos, que es lo que usa el Hamiltoniano de qubits. El chequeo de **estabilidad interna** (RHF → RHF) pasa en todos los puntos: no hay una solución RHF más baja que la encontrada.
- Tolerancias ajustadas (`1e-11` en SCF, `1e-10` en CCSD, `1e-12` en FCI). El código verifica la convergencia de RHF y CCSD en cada punto y que el estado FCI sea singlete (⟨S²⟩ = 0). Todos convergen con el `init_guess` por defecto, así que no hizo falta arrancar desde la densidad del punto vecino.

**Qué muestran los números**

- Cerca del equilibrio (0,6–1,1 Å) CCSD está a menos de 0,01 mHa de FCI.
- Desde ~1,1 Å CCSD **deja de ser variacional** y queda por debajo de FCI. El error es −9,1 mHa a 1,8 Å (el punto más exigente de la zona juzgada), llega a −30,6 mHa a 2,6 Å y vuelve a −23,0 mHa a 3,0 Å.
- Para ganarle a CCSD hace falta `|E_VQE − E_FCI| + σ < 9,1 mHa` a 1,8 Å y < 18 mHa desde 2,0 Å.

In [ ]:
# 1. Referencias clásicas: HF, CCSD y FCI en toda la grilla D (#1).
# Solo sirven para comparar y elegir puntos de partida; nunca son resultados.
# Las energías no dependen del signo de los orbitales: los orbitales "oficiales" (signo fijo,
# los del Hamiltoniano) son los de h4_scf(d) en la sección 2. Acá se usa el mismo procedimiento.
import os
import matplotlib.pyplot as plt
from pyscf import cc, fci, gto, scf


def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'} en Ha (energías totales)."""
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    # Al estirar, el SCF podría caer en un punto silla. Si la solución tiene una inestabilidad
    # interna (RHF -> RHF), seguimos la dirección inestable y reconvergemos (en esta grilla no pasa).
    for _ in range(5):
        mo, _, stable, _ = mf.stability(return_status=True)
        if stable:
            break
        mf.kernel(dm0=mf.make_rdm1(mo, mf.mo_occ))
    assert mf.converged, f"RHF no convergió en d={d}"

    ccsd = cc.CCSD(mf)
    ccsd.conv_tol, ccsd.max_cycle = 1e-10, 500
    ccsd.kernel()
    assert ccsd.converged, f"CCSD no convergió en d={d}"

    solver = fci.FCI(mf)
    solver.conv_tol = 1e-12
    e_fci, ci = solver.kernel()
    s2, _ = solver.spin_square(ci, N_ORB, (N_ALPHA, N_BETA))
    assert abs(s2) < 1e-6, f"FCI no es singlete en d={d}: <S²>={s2:.3g}"
    return {"d": float(d), "e_hf": mf.e_tot, "e_ccsd": ccsd.e_tot, "e_fci": e_fci}


refs = pd.DataFrame([references(d) for d in D])

# Chequeos de cordura: FCI es exacto en la base (variacional), CCSD es casi exacto cerca del equilibrio.
assert (refs.e_fci <= refs.e_hf + 1e-10).all()
near_eq = refs[(refs.d >= 0.8) & (refs.d <= 1.0)]
assert ((near_eq.e_ccsd - near_eq.e_fci).abs() < CHEM_ACC).all()

os.makedirs("results", exist_ok=True)
refs.to_csv("results/classical.csv", index=False)

err_ccsd = 1e3 * (refs.e_ccsd - refs.e_fci)  # mHa, con signo
print(refs.assign(ccsd_menos_fci_mHa=err_ccsd).to_string(
    index=False, formatters={"d": "{:.2f}".format, "ccsd_menos_fci_mHa": "{:+.3f}".format}, float_format="%.6f"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(refs.d, refs.e_hf, "s-", ms=4, label="RHF")
ax1.plot(refs.d, refs.e_ccsd, "^-", ms=4, label="CCSD")
ax1.plot(refs.d, refs.e_fci, "o-", ms=4, color="k", label="FCI (exacta)")
ax1.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0, label=f"zona estirada (d ≥ {STRETCHED} Å)")
ax1.set(xlabel="d (Å)", ylabel="E (Ha)", title="Curva de energía de H₄ (STO-3G)")
ax1.legend()

ax2.semilogy(refs.d, err_ccsd.abs(), "^-", ms=4, color="C1", label="|E_CCSD − E_FCI|")
ax2.axhline(1e3 * CHEM_ACC, ls="--", color="C2", label="precisión química (1,6 mHa)")
ax2.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0)
ax2.set(xlabel="d (Å)", ylabel="error (mHa)", title="Error de CCSD: lo que hay que superar")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Hamiltoniano de qubits

In [ ]:
import copy
from functools import lru_cache

from pyscf import ao2mo, fci, gto, scf
from pyscf.fci import cistring


def _so(p: int, spin: int) -> int:
    return alpha_qubit(p) if spin == 0 else beta_qubit(p)


@lru_cache(maxsize=None)
def _jw_ladder(n_qubits: int) -> tuple[tuple[SparsePauliOp, ...], tuple[SparsePauliOp, ...]]:
    annihilation = []
    for j in range(n_qubits):
        z_string = list(range(j))
        annihilation.append(SparsePauliOp.from_sparse_list(
            [("Z" * j + "X", z_string + [j], 0.5), ("Z" * j + "Y", z_string + [j], 0.5j)],
            num_qubits=n_qubits))
    creation = [a.adjoint() for a in annihilation]
    return tuple(creation), tuple(annihilation)


def jordan_wigner(h1_so: np.ndarray, h2_so: np.ndarray | None = None, atol: float = 1e-12,
                  hermitian: bool = True) -> SparsePauliOp:
    n = h1_so.shape[0]
    cdag, a = _jw_ladder(n)
    terms = [SparsePauliOp("I" * n, 0.0)]
    for p, q in np.argwhere(np.abs(h1_so) > atol):
        terms.append(h1_so[p, q] * cdag[p] @ a[q])
    if h2_so is not None:
        for p, q, r, s in np.argwhere(np.abs(h2_so) > atol):
            if p == r or q == s:
                continue
            terms.append(0.5 * h2_so[p, q, r, s] * cdag[p] @ cdag[r] @ a[s] @ a[q])
    op = SparsePauliOp.sum(terms).simplify(atol=atol)
    if not hermitian:
        return op
    if np.max(np.abs(op.coeffs.imag), initial=0.0) > 1e-10:
        raise ValueError("Operador no hermítico (coeficientes complejos): revisar índices o usar hermitian=False")
    return SparsePauliOp(op.paulis, op.coeffs.real)


def spatial_to_spin_orbitals(h1: np.ndarray, eri: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n_orb = h1.shape[0]
    n_so = 2 * n_orb
    h1_so = np.zeros((n_so, n_so))
    h2_so = np.zeros((n_so, n_so, n_so, n_so))
    for s1 in (0, 1):
        i = [_so(p, s1) for p in range(n_orb)]
        h1_so[np.ix_(i, i)] = h1
        for s2 in (0, 1):
            k = [_so(p, s2) for p in range(n_orb)]
            h2_so[np.ix_(i, i, k, k)] = eri
    return h1_so, h2_so


def _distance_key(d: float) -> float:
    return round(float(d), 6)


@lru_cache(maxsize=None)
def _h4_scf_cached(d: float) -> scf.hf.RHF:
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    if not mf.converged:
        raise RuntimeError(f"RHF no convergió en d = {d}")
    c = mf.mo_coeff
    significant = np.abs(c) > 1e-4
    if not significant.any(axis=0).all():
        raise RuntimeError(f"Orbital sin coeficientes > 1e-4 en d = {d}: no se puede fijar el signo")
    first = np.argmax(significant, axis=0)
    mf.mo_coeff = c * np.sign(c[first, np.arange(c.shape[1])])
    return mf


def h4_scf(d: float) -> scf.hf.RHF:
    mf = copy.copy(_h4_scf_cached(_distance_key(d)))
    for attr in ("mo_coeff", "mo_energy", "mo_occ"):
        setattr(mf, attr, getattr(mf, attr).copy())
    return mf


@lru_cache(maxsize=None)
def _qubit_hamiltonian_cached(d: float) -> tuple[SparsePauliOp, float]:
    mf = _h4_scf_cached(d)
    mol, c = mf.mol, mf.mo_coeff
    h1 = c.T @ mf.get_hcore() @ c
    eri = ao2mo.restore(1, ao2mo.full(mol, c), c.shape[1])
    h1_so, h2_so = spatial_to_spin_orbitals(h1, eri)
    return jordan_wigner(h1_so, h2_so), mol.energy_nuc()


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    op, e_nuc = _qubit_hamiltonian_cached(_distance_key(d))
    return op.copy(), e_nuc


_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])


def sector_ground_energy(op: SparsePauliOp) -> float:
    m = op.to_matrix()
    return np.linalg.eigvalsh(m[np.ix_(SECTOR, SECTOR)])[0].real


def hf_energy_from_op(op: SparsePauliOp) -> float:
    i = int(HF_BITSTRING, 2)
    return op.to_matrix()[i, i].real


def validate_hamiltonian(d: float) -> dict:
    op, e_nuc = qubit_hamiltonian(d)
    mf = h4_scf(d)
    e_fci = fci.FCI(mf).kernel()[0]
    e_hf_op = hf_energy_from_op(op) + e_nuc
    e_gs_op = sector_ground_energy(op) + e_nuc
    return {"d": float(d), "n_paulis": len(op), "e_hf": mf.e_tot, "e_fci": e_fci,
            "err_hf": abs(e_hf_op - mf.e_tot), "err_fci": abs(e_gs_op - e_fci)}


ham_check = pd.DataFrame([validate_hamiltonian(d) for d in D])
assert (ham_check[["err_hf", "err_fci"]] < 1e-8).all().all(), ham_check
ham_check


### Decisiones (#2)

**Firma.** `qubit_hamiltonian(d) -> (op, e_nuc)`, como dice el contrato 0.1 ([ADR-0001](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0001-firma-qubit-hamiltonian.md)). `op` es solo la parte electrónica: energía total = $\langle op \rangle + e_\text{nuc}$. El estado HF es la constante `HF_BITSTRING = "00110011"` porque no depende de $d$.

**Orden de qubits.** Qubit $p$ = espín-orbital $(p, \alpha)$, qubit $4+p$ = $(p, \beta)$, con $p = 0..3$ los orbitales RHF ordenados por energía. `HF_BITSTRING` deja ocupados q0, q1, q4, q5 (Qiskit es little-endian: el label se lee q7 … q0).

**Hamiltoniano.** Integrales en base MO de RHF: $h_{pq} = C^T h^\text{core} C$ y $(pq|rs)$ con `ao2mo` (notación química). En espín-orbitales:

$$\hat H = \sum_{PQ} h_{PQ}\, a_P^\dagger a_Q + \tfrac12 \sum_{PQRS} (PQ|RS)\, a_P^\dagger a_R^\dagger a_S a_Q,$$

con el espín conservado dentro de cada par $(P,Q)$ y $(R,S)$. Jordan-Wigner a mano: $a_j = Z_0 \cdots Z_{j-1}\,(X_j + iY_j)/2$ ([ADR-0002](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0002-jordan-wigner-a-mano.md)). `jordan_wigner(h1_so, h2_so)` es genérica: sirve también para $N$, $S_z$, $S^2$ (#8) y para el tapering (#14). Por defecto exige un operador hermítico (coeficientes reales) y falla si no lo es, porque eso delata un error de índices; para operadores no hermíticos como $S^+ = \sum_p a^\dagger_{p\alpha} a_{p\beta}$ usar `hermitian=False`, que devuelve los coeficientes complejos.

**Signo de los orbitales.** PySCF puede devolver un orbital con el signo invertido entre una distancia y la siguiente. El espectro no cambia, pero los coeficientes de $\hat H$ sí, y eso rompe el *warm start* de #5. Se fija el signo para que el primer coeficiente de cada orbital con $|c| > 10^{-4}$ sea positivo (si algún orbital no tiene ninguno, se lanza un error en vez de anularlo) ([ADR-0003](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0003-signo-orbitales-y-cache.md)). Verificado: ningún término de Pauli cambia de signo de golpe entre distancias vecinas (los únicos cambios, cerca de 0,67 Å, son coeficientes que cruzan cero de forma suave).

**Orbitales RHF en toda la grilla.** RHF es inestable hacia UHF desde ~1,1 Å (`docs/analysis/state-of-the-art.md`; verificado con `mf.stability(external=True)`), pero se mantienen los orbitales RHF: solo definen la base (FCI no cambia), el estado de referencia sigue siendo un singlete y ⟨S²⟩ = 0 sigue sirviendo como diagnóstico en la sección 6 (#8). Con orbitales UHF la referencia quedaría contaminada de espín.

**Caché.** RHF y el Hamiltoniano se calculan una sola vez por distancia (`lru_cache` en funciones privadas). Las funciones públicas devuelven **copias**: `qubit_hamiltonian` devuelve `op.copy()` y `h4_scf` una copia del `mf` con sus propios `mo_coeff`, `mo_energy` y `mo_occ`, así que modificar lo devuelto (o correr `mf.kernel()` / `mf.stability()` sobre él) no altera el caché ni desincroniza orbitales y Hamiltoniano. La clave del caché es `d` redondeada a 6 decimales: `qubit_hamiltonian(0.1 * 7)` reutiliza el de `0.7`.

**Validación** (tabla `ham_check`, las 19 distancias):

| Prueba | Criterio | Resultado |
|---|---|---|
| $\langle \text{HF}\vert op \vert \text{HF}\rangle + e_\text{nuc}$ vs. $E_\text{HF}$ de PySCF | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Mínimo de `op` en el sector $N=4$, $S_z=0$ (36 estados) + $e_\text{nuc}$ vs. FCI | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Términos de Pauli | — | 185 (184 más la identidad) en todas las distancias, mismo conjunto de Paulis |

`SECTOR` son los 36 índices de la base computacional con 2 α y 2 β, **en el orden `ci[ia, ib]` de PySCF** (`cistring.make_strings`): un vector CI `ci` de PySCF corresponde a `psi[SECTOR] = ci.ravel()` (signos verificados contra `fci.direct_spin1.energy` en estados al azar, error < 3e-14 Ha). Es la misma definición que usa la sección 3 (#4).

La validación contra FCI se hace en el sector $N=4$, $S_z=0$: el mínimo del espectro completo (256 estados) puede tener otro número de electrones. FCI se calcula acá mismo con los orbitales de `h4_scf`; cuando esté `refs` (#1) se puede comparar también contra `refs.e_fci`.

**Qué no funcionó.** La primera versión armaba los productos con `A.compose(B)`, que en Qiskit es la matriz $B \cdot A$, no $A \cdot B$: quedaba $a_Q a_P^\dagger$ en vez de $a_P^\dagger a_Q$, o sea el Hamiltoniano con partículas y huecos intercambiados. Con 4 electrones en 8 espín-orbitales (medio llenado) ese operador tiene **el mismo espectro en el sector** $N=4$, así que la prueba contra FCI pasaba igual; solo la prueba de HF lo detectó (el estado HF aparecía en `11001100`). Se usa `@` (`A @ B` = $A \cdot B$). Por eso las dos pruebas son necesarias.

## 3. Ansatz

### Decisión (#4)

Elegimos un **2-QUpCCGSD podado**: excitaciones de pares $(p_\uparrow p_\downarrow)\to(q_\uparrow q_\downarrow)$ y simples $p\to q$ (mismo ángulo en α y β), como excitaciones de **qubit** (sin cadenas Z), en dos capas, con 19 de las 24 excitaciones.

Comparación con statevector (error vs. FCI en mHa a 0,9 / 1,8 / 3,0 Å; `cz` tras transpilar a `cz, sx, x, rz` en línea):

| Ansatz | `cz` | Parámetros | Error (mHa) |
|---|---|---|---|
| UCCSD (fermiónico) | 1277 | 26 | 0,03 / 3,3 / 0,46 |
| UCCSD de qubits | 1247 | 26 | 0,003 / 3,0 / 0,21 |
| 1-UpCCGSD | 474 | 12 | 14 / 11 / 0,29 |
| 2-UpCCGSD (fermiónico) | 982 | 24 | exacto |
| 2-QUpCCGSD (Givens) | 590 | 24 | exacto |
| **2-QUpCCGSD podado (elegido)** | **465** | **19** | **≤ 0,03 en toda la grilla** |
| 2-QUpCCGSD podado más agresivo | 411 | 17 | ≤ 0,5 (en 6 distancias) |
| QEB-ADAPT-VQE (un circuito por distancia) | 461–573 | 16–19 | ≤ 0,01 |
| Hardware-efficient (RY·RZ + CZ en línea), 6 capas | 42 | 112 | 53 / 23 / 0,67 |
| Hardware-efficient, 8 capas (optimizando fidelidad con FCI) | 56 | 144 | 1 − F = 1e-5 / 1,5e-3 / 2e-4 |
| *CCSD (referencia)* | – | – | 0,007 / −9,1 / −23 |

Por qué:

- **Llega a FCI en la zona estirada.** UCCSD falla ahí (3,3 mHa a 1,8 Å, fuera de precisión química) por la misma razón que CCSD: el estado se vuelve multirreferencia. Las excitaciones generalizadas de pares sí lo capturan.
- **Conserva N = 4 y S_z = 0**: cada excitación mueve electrones dentro del mismo bloque de espín. Lo verificamos numéricamente abajo, y sirve para el argumento de confiabilidad (#8).
- **Menos `cz`**: cada excitación es una rotación de Givens entre dos estados de la base, implementada con un multiplexor de RY en código Gray (14 CX por doble) en lugar de `PauliEvolutionGate` (837 → 590 `cz`). Después podamos de a una las excitaciones de menor ángulo mientras el error siguiera < 0,1 mHa en 6 distancias (590 → 465 `cz`).

Qué no funcionó o descartamos:

- UCCSD y 1-UpCCGSD: no llegan a precisión química en la zona estirada.
- Sintetizar las excitaciones con `PauliEvolutionGate`: correcto pero caro en `cz`.
- **Hardware-efficient** (RY·RZ por qubit + cadena de CZ): muy barato en `cz` (42–56), pero optimizando la energía queda trabado lejos de FCI (53 mHa a 0,9 Å y 23 mHa a 1,8 Å con 6 capas y 6 arranques). Con 8 capas *puede* representar el estado (fidelidad 1 − 1,5e-3 a 1,8 Å si se optimiza contra el vector FCI, cosa que en un VQE real no se puede), así que el problema es el paisaje de optimización, no la expresividad. Además no conserva N ni S_z (fuga de hasta 0,4 %), lo que complica #8. Queda como opción para los extras con ruido si alguien encuentra cómo optimizarlo.
- **QEB-ADAPT-VQE** (sugerido en #4): con un pool de 90 excitaciones de qubit converge a ≤ 0,01 mHa con 16–19 excitaciones, pero necesita 461–573 `cz` (sus dobles αβ genéricas tocan qubits lejanos en la línea) y un circuito distinto por distancia. Incluso cortado justo al llegar a precisión química necesita 467 `cz` a 1,8 Å. No mejora al ansatz elegido.
- La versión de 17 excitaciones (411 `cz`) queda como alternativa si los extras con ruido necesitan menos compuertas, a costa de errores de hasta 0,5 mHa.

Limitaciones:

- **El paisaje de energía tiene mínimos locales.** Con un solo arranque, algunas distancias quedan a 1 mHa o hasta decenas de mHa; con 30 arranques el mejor llega a ≤ 0,003 mHa. Con los 10 arranques de la validación de abajo, el peor punto queda entre 0,04 y 0,13 mHa según la corrida (diferencias numéricas mínimas cambian a qué mínimo cae cada arranque). La optimización (#5) tiene que usar multi-start.
- **Contaminación de espín.** Conservar N y S_z no garantiza S² = 0: las excitaciones de qubit no están adaptadas al espín, así que si la optimización cae en otro mínimo el estado puede mezclarse con el triplete, que en la zona estirada está casi degenerado (brecha de 0,5 mHa a 3,0 Å, así que la energía no lo delata). Con el Hamiltoniano de #2 (orbitales con signo fijo) la validación llega al singlete: ⟨S²⟩ ≤ 3e-7. Con un stand-in previo cuyos orbitales no tenían el signo fijo, el warm start entre distancias caía en otros mínimos y vimos ⟨S²⟩ de hasta 3e-2 (~1,5 % de triplete) desde 2,0 Å. Por eso la tabla de abajo reporta ⟨S²⟩ y la brecha, #8 tiene que reportarlo, y #5 puede asegurarlo con una penalización λ⟨S²⟩.
- **465 `cz` es mucho para hardware ruidoso.** Con 0,1 % de error por `cz`, la probabilidad de al menos una falla es 1 − 0,999⁴⁶⁵ ≈ 37 %. Para el desafío central (sin ruido) no importa; para los extras con ruido conviene la versión podada más agresiva o menos qubits (#14).

La validación de abajo usa el Hamiltoniano de #2 (`qubit_hamiltonian`) restringido a los 36 estados con N = 4 y S_z = 0, y toma como FCI su autovalor mínimo. Mientras #2 no estaba, la exploración usó un stand-in armado con PySCF (misma convención de signos, verificado en #4); con el Hamiltoniano de #2 los resultados son los mismos.


In [ ]:
# Ansatz elegido (#4): 2-QUpCCGSD podado, 19 excitaciones de qubit como rotaciones de Givens.
import itertools
from qiskit.circuit import ParameterVector


def givens_single(qc: QuantumCircuit, t, i: int, k: int) -> None:
    """exp(θ(Q†_k Q_i − h.c.)): rota |q_i=1, q_k=0⟩ hacia |q_i=0, q_k=1⟩."""
    qc.cx(k, i)
    qc.cry(2 * t, i, k)
    qc.cx(k, i)


def _ucry(qc: QuantumCircuit, t, controls: list, target: int, pattern: tuple) -> None:
    """RY(2θ) en target solo cuando controls == pattern. Multiplexor Gray: 2^n CX."""
    n = 2 ** len(controls)
    gray = [m ^ (m >> 1) for m in range(n)]
    idx = sum(b << c for c, b in enumerate(pattern))
    for m in range(n):
        qc.ry(2 * t * (-1) ** bin(idx & gray[m]).count("1") / n, target)
        qc.cx(controls[(gray[m] ^ gray[(m + 1) % n]).bit_length() - 1], target)


def givens_double(qc: QuantumCircuit, t, i: int, j: int, k: int, l: int) -> None:
    """exp(θ(Q†_k Q†_l Q_j Q_i − h.c.)): rota |ij=11, kl=00⟩ hacia |ij=00, kl=11⟩ (14 CX)."""
    for m in (i, j, l):
        qc.cx(k, m)
    _ucry(qc, t, [i, j, l], k, (1, 1, 0))
    for m in (l, j, i):
        qc.cx(k, m)


# (capa, tipo, p, q). "D": par (p↑, p↓) → (q↑, q↓). "S": simple p → q, en α y en β con el mismo θ.
ANSATZ_OPS = [
    (0, "D", 0, 2), (0, "D", 1, 2), (0, "D", 1, 3), (0, "D", 2, 3),
    (0, "S", 0, 1), (0, "S", 0, 3), (0, "S", 1, 2), (0, "S", 1, 3), (0, "S", 2, 3),
    (1, "D", 0, 1), (1, "D", 0, 3), (1, "D", 1, 2), (1, "D", 1, 3), (1, "D", 2, 3),
    (1, "S", 0, 2), (1, "S", 0, 3), (1, "S", 1, 2), (1, "S", 1, 3), (1, "S", 2, 3),
]


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito de 8 qubits que arranca en HF_BITSTRING; un parámetro por excitación."""
    th = ParameterVector("θ", len(ANSATZ_OPS))
    qc = QuantumCircuit(N_QUBITS, name="QUpCCGSD-19")
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for t, (_, kind, p, q) in zip(th, ANSATZ_OPS):
        if kind == "D":
            givens_double(qc, t, alpha_qubit(p), beta_qubit(p), alpha_qubit(q), beta_qubit(q))
        else:
            givens_single(qc, t, alpha_qubit(p), alpha_qubit(q))
            givens_single(qc, t, beta_qubit(p), beta_qubit(q))
    return qc


ansatz = build_ansatz()
print(f"{ansatz.name}: {ansatz.num_parameters} parámetros, {cz_count(ansatz)} cz (transpilado a cz/sx/x/rz en línea)")


In [ ]:
# Validación del ansatz en toda la grilla, con statevector (permitido para elegir parámetros y ansatz).
# Usa el Hamiltoniano de #2 (`qubit_hamiltonian`) restringido al sector N=4, S_z=0 (36 estados):
# el ansatz conserva N y S_z, así que simular ahí es exacto y mucho más rápido que con 256 amplitudes.
from pyscf.fci import cistring, spin_op
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector

_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])  # índices de qubits


def sector_hamiltonian(d: float) -> tuple[np.ndarray, float]:
    """H total (con e_nuc) de #2 en el sector de 36 estados, y su autovalor mínimo (= E_FCI)."""
    op, e_nuc = qubit_hamiltonian(d)
    M = op.to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    assert np.abs(M.imag).max() < 1e-12
    H = M.real + e_nuc * np.eye(len(SECTOR))
    return H, np.linalg.eigvalsh(H)[0]


def _qubit_excitation(create: tuple, annihilate: tuple) -> np.ndarray:
    """K = T − T† en el sector, con T = Q†...Q... (operadores de qubit, sin cadenas Z)."""
    def q(j, dag):
        lab = lambda p: "I" * (N_QUBITS - j - 1) + p + "I" * j
        return SparsePauliOp([lab("X"), lab("Y")], [0.5, -0.5j if dag else 0.5j])
    T = SparsePauliOp("I" * N_QUBITS)
    for c in create:
        T = T @ q(c, True)
    for a in annihilate:
        T = T @ q(a, False)
    M = (T - T.adjoint()).simplify().to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    return M.real


class SectorAnsatz:
    """El mismo ansatz simulado en el sector de 36 estados, con gradiente adjunto exacto (rápido)."""

    def __init__(self, ops):
        self.groups = []
        for _, kind, p, q in ops:
            if kind == "D":
                self.groups.append([_qubit_excitation((alpha_qubit(q), beta_qubit(q)), (beta_qubit(p), alpha_qubit(p)))])
            else:
                self.groups.append([_qubit_excitation((alpha_qubit(q),), (alpha_qubit(p),)),
                                    _qubit_excitation((beta_qubit(q),), (beta_qubit(p),))])
        self.n = len(self.groups)
        self.hf = (SECTOR == int(HF_BITSTRING, 2)).astype(float)

    def _U(self, k, t):  # cada generador cumple K³ = −K, así que exp(tK) = 1 + sin(t)K + (1 − cos(t))K²
        I = np.eye(len(SECTOR))
        U = I
        for K in self.groups[k]:
            U = (I + np.sin(t) * K + (1 - np.cos(t)) * K @ K) @ U
        return U

    def state(self, x):
        v = self.hf
        for k in range(self.n):
            v = self._U(k, x[k]) @ v
        return v

    def energy_grad(self, x, H):
        Us = [self._U(k, x[k]) for k in range(self.n)]
        v = self.hf
        for U in Us:
            v = U @ v
        lam = H @ v
        e, g = v @ lam, np.empty(self.n)
        for k in reversed(range(self.n)):
            g[k] = 2 * lam @ (sum(self.groups[k]) @ v)
            v, lam = Us[k].T @ v, Us[k].T @ lam
        return e, g


sector_ansatz = SectorAnsatz(ANSATZ_OPS)

# El circuito real y la simulación en el sector tienen que dar el mismo estado.
_x = np.random.default_rng(0).normal(size=ansatz.num_parameters)
_full = Statevector(ansatz.assign_parameters(_x)).data
assert np.isclose(np.linalg.norm(_full[SECTOR]), 1), "el ansatz se sale del sector N=4, S_z=0"
assert np.isclose(abs(np.vdot(_full[SECTOR], sector_ansatz.state(_x))), 1), "circuito ≠ simulación en el sector"

# El paisaje de energía tiene mínimos locales (algunos arranques quedan a decenas de mHa): por eso
# probamos varios arranques por distancia (cero, el óptimo de la distancia anterior y 8 al azar).
rng = np.random.default_rng(0)
rows, x0 = [], np.zeros(sector_ansatz.n)
for d in D:
    H, e_fci = sector_hamiltonian(d)
    starts = [np.zeros(sector_ansatz.n), x0] + [rng.normal(scale=0.7, size=sector_ansatz.n) for _ in range(8)]
    best = min((minimize(sector_ansatz.energy_grad, s, args=(H,), jac=True, method="BFGS", options={"gtol": 1e-10})
                for s in starts), key=lambda r: r.fun)
    x0 = best.x
    s2 = spin_op.spin_square0(sector_ansatz.state(best.x).reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0]
    gap = np.diff(np.linalg.eigvalsh(H)[:2])[0]
    rows.append({"d": d, "e_ansatz": best.fun, "e_fci": e_fci, "err_mHa": (best.fun - e_fci) * 1e3,
                 "S2": s2, "gap_mHa": gap * 1e3})
ansatz_check = pd.DataFrame(rows)
print(f"error máximo vs. FCI en la grilla: {ansatz_check.err_mHa.abs().max():.3f} mHa "
      f"(zona estirada: {ansatz_check[ansatz_check.d >= STRETCHED].err_mHa.abs().max():.3f} mHa); "
      f"⟨S²⟩ máximo: {ansatz_check.S2.max():.1e} (singlete puro = 0, triplete = 2)")
ansatz_check


## 4. VQE

In [ ]:
# VQE, parte 1 (#5): optimización de θ*(d) en toda la grilla, con statevector (permitido para buscar
# parámetros). Las energías de esta celda son solo diagnóstico: la energía reportada la mide #6.
# Reutiliza de la sección 3 `sector_hamiltonian`, `sector_ansatz` (gradiente adjunto exacto), `ansatz`,
# `minimize`, `Statevector` y `spin_op`.
import time

_t0 = time.perf_counter()
H_SECTOR = {float(d): sector_hamiltonian(d) for d in D}  # d -> (H con e_nuc, E_FCI); E_FCI solo para comparar

# S² en el sector (mismo orden ci[ia, ib] que SECTOR). Se minimiza E(θ) + λ⟨S²⟩(θ): ⟨S²⟩ es otro valor
# esperado, así que el gradiente adjunto sirve igual con H + λS². En el singlete la penalización vale 0
# y no mueve el mínimo; sin ella, en la zona estirada hay mínimos locales contaminados con triplete a
# solo ~0,02 mHa de FCI (la brecha singlete-triplete es de 0,5 mHa a 3,0 Å), que la energía no delata.
S2_SECTOR = np.column_stack([spin_op.contract_ss(e.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA)).ravel()
                             for e in np.eye(len(SECTOR))])
_v = sector_ansatz.state(np.random.default_rng(2).normal(size=sector_ansatz.n))
assert np.allclose(S2_SECTOR, S2_SECTOR.T)
assert np.isclose(_v @ S2_SECTOR @ _v, spin_op.spin_square0(_v.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0])
LAMBDA_S2 = 0.05  # Ha: ⟨S²⟩ = 0,03 (1,5 % de triplete) pasa a costar 1,5 mHa
COST = {d: H + LAMBDA_S2 * S2_SECTOR for d, (H, _) in H_SECTOR.items()}

# El gradiente adjunto de la sección 3 coincide con diferencias finitas centradas.
_x, _C = np.random.default_rng(1).normal(size=sector_ansatz.n), COST[STRETCHED]
_fd = [(sector_ansatz.energy_grad(_x + h, _C)[0] - sector_ansatz.energy_grad(_x - h, _C)[0]) / 2e-6
       for h in 1e-6 * np.eye(sector_ansatz.n)]
assert np.allclose(sector_ansatz.energy_grad(_x, _C)[1], _fd, atol=1e-7), "gradiente adjunto ≠ diferencias finitas"


def optimize(d: float, x0: np.ndarray):
    """BFGS con gradiente adjunto exacto sobre E + λ⟨S²⟩. |∇| < 1e-6 deja el costo a ~1e-12 Ha del mínimo local."""
    return minimize(sector_ansatz.energy_grad, x0, args=(COST[d],), jac=True, method="BFGS", options={"gtol": 1e-6})


def diagnostics(d: float, x: np.ndarray) -> dict:
    """Energía statevector (sin penalización), error vs. FCI y ⟨S²⟩ (0 = singlete, 2 = triplete)."""
    v = sector_ansatz.state(x)
    H, e_fci = H_SECTOR[d]
    return {"e_sv": v @ H @ v, "err_mHa": (v @ H @ v - e_fci) * 1e3, "S2": v @ S2_SECTOR @ v}


# 1) Warm start en las dos direcciones + multi-start: en cada d se arranca desde θ* de la distancia vecina,
#    desde N_PERTURB perturbaciones de ese θ* y desde N_RANDOM puntos al azar; se queda el mejor.
N_PERTURB, N_RANDOM = 2, 3
rng = np.random.default_rng(5)
best, starts_log = {}, []  # best: d -> (resultado de minimize, origen)
for direction, order in (("0,6→3,0", D), ("3,0→0,6", D[::-1])):
    x = np.zeros(sector_ansatz.n)  # θ = 0 es HF: el primer punto se salva con los arranques al azar
    for d in map(float, order):
        starts = ([("warm", x)] + [("warm+ruido", x + rng.normal(scale=0.3, size=x.size)) for _ in range(N_PERTURB)]
                  + [("azar", rng.normal(scale=0.7, size=x.size)) for _ in range(N_RANDOM)])
        runs = []
        for kind, x0 in starts:
            r = optimize(d, x0)
            runs.append((r, f"{direction} {kind}"))
            starts_log.append({"d": d, "direction": direction, "start": kind, **diagnostics(d, r.x),
                               "nit": r.nit, "converged": r.success})
        r, origin = min(runs, key=lambda t: t[0].fun)
        x = r.x
        if d not in best or r.fun < best[d][0].fun:
            best[d] = (r, origin)
starts_log = pd.DataFrame(starts_log)

# 2) Pulido: cada d se reoptimiza desde el θ* final de sus dos vecinos, hasta que nada mejore.
#    Corrige los puntos donde ninguna de las dos pasadas encontró el mismo mínimo que sus vecinos.
#    Cada ronda recorre la grilla en el sentido opuesto a la anterior, así una mejora se propaga entera.
grid = list(map(float, D))
for n_polish in range(1, 21):
    improved = 0
    for i in (range(len(grid)) if n_polish % 2 else reversed(range(len(grid)))):
        for j in (i - 1, i + 1):
            if 0 <= j < len(grid):
                r = optimize(grid[i], best[grid[j]][0].x)
                if r.fun < best[grid[i]][0].fun - 1e-9:
                    best[grid[i]], improved = (r, f"pulido desde {grid[j]}"), improved + 1
    if not improved:
        break
assert not improved, "el pulido no terminó en 20 rondas"

params = {d: best[d][0].x for d in grid}  # contrato 0.1: d -> θ* (np.ndarray, en el orden de build_ansatz)
opt = pd.DataFrame([{"d": d, **diagnostics(d, r.x), "e_fci": H_SECTOR[d][1], "origin": origin, "nit": r.nit,
                     "nfev": r.nfev, "converged": r.success, "grad_norm": np.linalg.norm(r.jac)}
                    for d, (r, origin) in best.items()]).sort_values("d", ignore_index=True)
t_opt = time.perf_counter() - _t0

# 3) Paisaje en la zona estirada (para #8): resultado desde 10 arranques distintos (HF, warm desde la
#    distancia vecina ya optimizada y 8 al azar), cada uno por separado.
MULTISTART_D = (1.8, 2.4, 3.0)
ms_rows = []
for d in MULTISTART_D:
    neighbor = grid[grid.index(d) - 1]
    starts = ([("HF (θ=0)", np.zeros(sector_ansatz.n)), (f"warm desde {neighbor}", params[neighbor])]
              + [(f"azar {i}", rng.normal(scale=0.7, size=sector_ansatz.n)) for i in range(8)])
    for kind, x0 in starts:
        r = optimize(d, x0)
        ms_rows.append({"d": d, "start": kind, **diagnostics(d, r.x), "nit": r.nit, "converged": r.success})
multistart = pd.DataFrame(ms_rows)


def vqe_state(d: float) -> Statevector:
    """Statevector de 8 qubits del circuito real en θ*(d) (para ⟨S²⟩ y la varianza en #8)."""
    return Statevector(ansatz.assign_parameters(params[float(d)]))


# El circuito real con el Hamiltoniano completo de 8 qubits da la misma energía que la simulación en el sector.
for d in (0.9, STRETCHED, 2.4, DISSOCIATED):
    op, e_nuc = qubit_hamiltonian(d)
    e_full = vqe_state(d).expectation_value(op).real + e_nuc
    assert abs(e_full - opt.set_index("d").e_sv[d]) < 1e-9, (d, e_full)

stretched = opt[opt.d >= STRETCHED]
assert len(params) == len(D) and opt.converged.all()
assert (opt.S2 < 1e-3).all(), opt  # singlete en todas las distancias (#8)
assert (stretched.err_mHa.abs() < CHEM_ACC * 1e3).all(), stretched
jump = np.abs(np.diff(opt.err_mHa)).max()  # suavidad: un salto del error entre vecinos delata un mínimo local
print(f"{time.perf_counter() - _t0:.0f} s ({t_opt:.0f} s optimización, {n_polish} rondas de pulido). "
      f"Error máx. vs. FCI: {opt.err_mHa.abs().max():.4f} mHa (zona estirada: {stretched.err_mHa.abs().max():.4f} mHa); "
      f"salto máx. del error entre vecinos: {jump:.4f} mHa; ⟨S²⟩ máx.: {opt.S2.max():.1e}")
print("Multi-start en la zona estirada (10 arranques por distancia, sin elegir):")
print(multistart.groupby("d").agg(err_min=("err_mHa", "min"), err_mediana=("err_mHa", "median"),
                                  err_max=("err_mHa", "max"), S2_max=("S2", "max")).to_string(float_format="%.3g"))
opt

### Decisiones (#5)

**Qué deja esta celda.** `params = {d: θ*}` (contrato 0.1, un `np.ndarray` de 19 ángulos por distancia, en el orden de `build_ansatz`), la tabla `opt` (energía statevector, error vs. FCI, ⟨S²⟩, iteraciones, convergencia y de qué arranque salió cada θ*), `starts_log` (todos los arranques del barrido), `multistart` (10 arranques sin elegir a 1,8 / 2,4 / 3,0 Å) y `vqe_state(d)`, que regenera el statevector de 8 qubits del circuito real en θ*(d) para ⟨S²⟩ y la varianza (#8). Las energías de `opt` son **solo diagnóstico**: la tabla `vqe` del contrato (energía, σ, shots, `cz`) la arma #6 midiendo con `qiskit_aer`, nunca desde el statevector.

**Resultado** (una corrida, semilla fija):

| | Valor |
|---|---|
| Error máximo vs. FCI, toda la grilla | 0,003 mHa (exacto a < 1e-7 mHa hasta 2,2 Å) |
| Error máximo vs. FCI, $d \geq 1{,}8$ Å | 0,003 mHa (criterio: 1,6 mHa) |
| Salto máximo del error entre distancias vecinas | 0,002 mHa (curva suave) |
| ⟨S²⟩ máximo | 2e-5 (singlete) |
| Convergencia | 19/19 (BFGS, $\lvert\nabla\rvert < 10^{-6}$), 45–140 iteraciones por punto |
| Tiempo de la celda | 60–75 s local (~2,5 min el notebook entero); sin caché |

Verificado: el circuito real (`ansatz` con θ*) y el Hamiltoniano de 8 qubits de #2 dan la misma energía que la simulación en el sector (< 1e-9 Ha, a 0,9 / 1,8 / 2,4 / 3,0 Å).

**Simulador.** La búsqueda usa `sector_ansatz` de la sección 3: el ansatz conserva N y S_z, así que simular en los 36 estados del sector es exacto y mucho más rápido que con 256 amplitudes.

**Optimizador: BFGS con gradiente adjunto exacto.** El gradiente completo sale de una pasada hacia adelante y otra hacia atrás sobre el estado (verificado contra diferencias finitas). La regla de *parameter-shift* de 2 términos no sirve acá: el generador de cada excitación tiene autovalores {−1, 0, 1} (frecuencias 1 y 2: 4 evaluaciones con la regla generalizada) y las simples comparten ángulo en α y β (autovalores −2…2: 8 evaluaciones), o sea 9·4 + 10·8 = 116 circuitos por gradiente. Además cada θ aparece en varias compuertas (8 `ry` por doble). Con statevector, el gradiente adjunto es exacto y cuesta del orden de una evaluación de la energía.

**Warm start en las dos direcciones + multi-start + pulido.**

1. Se recorre la grilla 0,6 → 3,0 y 3,0 → 0,6. En cada $d$ se arranca desde θ* de la distancia anterior (*warm*), desde 2 perturbaciones de ese θ* (ruido gaussiano σ = 0,3) y desde 3 puntos al azar (σ = 0,7), y se queda el de menor costo. Entre las dos pasadas se guarda el mejor por punto.
2. Pulido: cada $d$ se reoptimiza desde el θ* final de sus dos vecinos, alternando el sentido del recorrido, hasta que ningún punto mejore (> 1e-9 Ha; 3 rondas).

Cada pieza aporta: el mejor por punto de una sola dirección queda a 0,06 mHa (ida) o 0,11 mHa (vuelta); combinando las dos y puliendo, a 0,003 mHa. Por tipo de arranque, en todo el barrido: desde *warm* el 95 % de las corridas llega a precisión química y el 47 % a < 0,01 mHa; desde el azar, el 75 % y el 10 %.

**Penalización de espín: se minimiza $E(\theta) + \lambda \langle S^2\rangle(\theta)$, con λ = 0,05 Ha.** ⟨S²⟩ es otro valor esperado, así que el gradiente adjunto se aplica a $H + \lambda S^2$ sin cambios. En el singlete exacto la penalización vale 0 y el mínimo no se mueve; `opt.e_sv` es la energía sin penalizar. Sin ella, con otra semilla el barrido terminó con ⟨S²⟩ = 3e-2 a 2,8 Å (≈ 1,5 % de triplete) y un error de solo 0,02 mHa: el triplete está a 0,5 mHa a 3,0 Å, así que la energía no lo delata (ver la limitación de la sección 3). Con la penalización, en tres semillas ⟨S²⟩ ≤ 2e-4 y el error ≤ 0,03 mHa.

**Multi-start en la zona estirada** (`multistart`, error vs. FCI en mHa, 10 arranques por distancia):

| $d$ (Å) | HF (θ = 0) | *warm* desde el vecino | mejor de 8 al azar | mediana de 8 al azar | peor de 8 al azar |
|---|---|---|---|---|---|
| 1,8 | 153 | < 1e-8 | 0,006 | 0,3 | 63 |
| 2,4 | 230 | 0,007 | 0,02 | 2,7 | 10,5 |
| 3,0 | 287 | 0,002 | 0,0002 | 0,12 | 1,3 |

**Qué no funcionó.**

- **Arrancar desde HF (θ = 0)**: cae en un punto estacionario a 10 mHa (0,6 Å) y hasta 287 mHa (3,0 Å). Por eso el primer punto de cada pasada depende de los arranques al azar.
- **Un solo arranque**: uno al azar queda fuera de precisión química en ~25 % de las corridas (hasta decenas de mHa) e incluso uno *warm* en ~5 %, como ya había visto #4.
- **L-BFGS-B**: llega a los mismos mínimos, pero tarda 2–4× más (con `ftol` relativo de 1e-13 necesita 300–900 iteraciones). **Ajustar la tolerancia** (`gtol` de 1e-6 a 1e-10, o L-BFGS-B con `ftol = 1e-15`) desde θ* no baja el error ni un µHa: los errores residuales de ~1e-3 mHa son mínimos locales reales, no falta de convergencia.
- **Reescribir el gradiente con productos matriz-vector** (en vez de armar cada $e^{\theta K}$): solo 15 % más rápido, porque domina el costo de Python; se reutiliza el de la sección 3.

**Limitaciones.** El paisaje tiene mínimos locales hasta el nivel del µHa, así que los últimos dígitos de `params` dependen de la semilla (el error se mantiene ≤ 0,04 mHa en todas las semillas probadas, con y sin penalización). Diferencias numéricas mínimas (del orden de 1e-13 Ha) cambian a qué mínimo cae un arranque al azar: entre dos corridas del mismo código, el mejor de los 8 al azar a 3,0 Å pasó de 0,0002 a 0,002 mHa. Elegir por $E + \lambda\langle S^2\rangle$ evita que uno de esos empates elija un mínimo contaminado (queda λ⟨S²⟩ por encima); entre mínimos singlete solo cambian los µHa. Por lo mismo, θ* cambia entre corridas aunque la energía casi no: para comparar corridas usar energías y ⟨S²⟩, no los ángulos.

## 5. Resultados: curva, $R_e$, $D_e$ y tabla (energía, $\sigma$, shots, `cz`)

## 6. Zona estirada ($d \geq 1{,}8$ Å) vs. CCSD

## Resultados clave

Resumen de una pantalla, recalculado desde las tablas del contrato `refs` (#1) y `vqe` (#6), sin números pegados a mano: $R_e$ y $D_e$ (VQE contra FCI y CCSD), error máximo en la zona estirada, si le ganamos a CCSD con la barra de error incluida, precisión química, `cz` por circuito y shots totales.

- $R_e$ sale de un ajuste cúbico local (5 puntos de la grilla alrededor del mínimo; para VQE, pesado con $1/\sigma$) y $D_e = E(3{,}0\ \text{Å}) - E(R_e)$ con $E(R_e)$ del mismo ajuste. Es una estimación rápida, igual para los tres métodos; los valores con incertidumbre (bootstrap) son los de la sección 5.
- "Le ganamos a CCSD" se evalúa punto por punto para $d \geq 1{,}8$ Å con el criterio del enunciado: $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$.
- Si `refs` o `vqe` todavía no existen, están incompletas o son un *stand-in* (marcado con `tabla.attrs["mock"] = True`), la celda lo dice y no calcula nada: nunca muestra números de mentira como si fueran resultados.

In [ ]:
# Resultados clave, recalculados desde `refs` (#1) y `vqe` (#6). Si faltan, avisa en vez de fallar.
from IPython.display import Markdown, display

_COLS = {"refs": ["d", "e_hf", "e_ccsd", "e_fci"], "vqe": ["d", "energy", "sigma", "shots", "cz"]}


def contract_table(name: str) -> tuple[pd.DataFrame | None, str | None]:
    """(tabla, None) si `name` cumple el contrato 0.1 en toda la grilla D; si no, (None, motivo)."""
    t = globals().get(name)
    if t is None:
        return None, f"`{name}` todavía no está definida"
    if getattr(t, "attrs", {}).get("mock"):
        return None, f"`{name}` es un stand-in (`attrs['mock']`), no un resultado"
    try:
        t = pd.DataFrame(t)
    except (TypeError, ValueError):
        return None, f"`{name}` no se puede leer como tabla"
    missing = [c for c in _COLS[name] if c not in t.columns]
    if missing:
        return None, f"a `{name}` le faltan las columnas {missing}"
    t = t[_COLS[name]].assign(d=lambda x: x.d.astype(float).round(2))
    absent = sorted(set(D) - set(t.d))
    if absent:
        return None, f"a `{name}` le faltan las distancias {', '.join(f'{x:g}' for x in absent)} Å"
    t = t[t.d.isin(D)].drop_duplicates("d").sort_values("d").reset_index(drop=True)
    if t.isna().any().any():
        return None, f"`{name}` tiene valores vacíos (NaN)"
    return t, None


def local_minimum(d: np.ndarray, e: np.ndarray, sigma: np.ndarray | None = None) -> tuple[float, float]:
    """(R_e, E(R_e)): mínimo de un ajuste cúbico a los 5 puntos de la grilla más cercanos al mínimo."""
    near = np.argsort(np.abs(d - d[np.argmin(e)]))[:5]
    w = None if sigma is None else 1 / np.maximum(sigma[near], 1e-12)
    poly = np.poly1d(np.polyfit(d[near], e[near], 3, w=w))
    x = np.linspace(d[near].min(), d[near].max(), 4001)
    return x[np.argmin(poly(x))], poly(x).min()


def key_results(refs: pd.DataFrame, vqe: pd.DataFrame) -> None:
    t = refs.merge(vqe, on="d")
    t["err_vqe"] = (t.energy - t.e_fci).abs() * 1e3   # mHa
    t["err_ccsd"] = (t.e_ccsd - t.e_fci).abs() * 1e3  # mHa
    t["sigma_mHa"] = t.sigma * 1e3
    t["gana_a_ccsd"] = t.err_vqe + t.sigma_mHa < t.err_ccsd
    s = t[t.d >= STRETCHED]

    d = t.d.to_numpy()
    e_end = t.set_index("d")
    curves = {"VQE": (t.energy.to_numpy(), t.sigma.to_numpy()), "FCI": (t.e_fci.to_numpy(), None),
              "CCSD": (t.e_ccsd.to_numpy(), None)}
    col = {"VQE": "energy", "FCI": "e_fci", "CCSD": "e_ccsd"}
    rows = {}
    for name, (e, sig) in curves.items():
        r_e, e_min = local_minimum(d, e, sig)
        rows[name] = {"R_e (Å)": r_e, "D_e (mHa)": (e_end.loc[DISSOCIATED, col[name]] - e_min) * 1e3}
    display(pd.DataFrame(rows).T.round({"R_e (Å)": 3, "D_e (mHa)": 1}))

    worst = s.loc[s.err_vqe.idxmax()]
    margin = s.loc[(s.err_ccsd - s.err_vqe - s.sigma_mHa).idxmin()]
    yes = lambda ok: "**sí**" if ok else "**no**"
    cz = sorted(t.cz.astype(int).unique())
    lo, hi = int(t.shots.min()), int(t.shots.max())
    shots_range = f"{lo:,}" if lo == hi else f"{lo:,}–{hi:,}"
    display(Markdown("\n".join([
        f"- **Error máximo en la zona estirada** ($d \\geq {STRETCHED}$ Å): {worst.err_vqe:.2f} mHa a {worst.d} Å "
        f"($\\sigma$ = {worst.sigma_mHa:.2f} mHa); CCSD ahí erra {worst.err_ccsd:.1f} mHa.",
        f"- **¿Le ganamos a CCSD?** {yes(s.gana_a_ccsd.all())}: en {int(s.gana_a_ccsd.sum())} de {len(s)} puntos "
        f"$|E_\\text{{VQE}} - E_\\text{{FCI}}| + \\sigma < |E_\\text{{CCSD}} - E_\\text{{FCI}}|$. Margen más chico: "
        f"{margin.err_ccsd - margin.err_vqe - margin.sigma_mHa:.2f} mHa a {margin.d} Å.",
        f"- **¿Precisión química** ($< {CHEM_ACC * 1e3:g}$ mHa)? Zona estirada: {yes((s.err_vqe < CHEM_ACC * 1e3).all())} "
        f"({int((s.err_vqe < CHEM_ACC * 1e3).sum())} de {len(s)} puntos); toda la curva: "
        f"{int((t.err_vqe < CHEM_ACC * 1e3).sum())} de {len(t)} puntos.",
        f"- **`cz` por circuito:** {', '.join(map(str, cz))} (transpilado a `cz, sx, x, rz` en línea).",
        (f"- **Shots:** {int(t.shots.sum()):,} en total para {len(t)} puntos ({shots_range} por punto)."
         .replace(",", ".")),
    ])))


_tables = {name: contract_table(name) for name in _COLS}
_pending = [why for _, why in _tables.values() if why]
if _pending:
    display(Markdown("**Resultados pendientes.** " + "; ".join(_pending) + ". "
                     "Esta celda se completa sola cuando las secciones 1 y 4 llenen `refs` y `vqe`."))
else:
    key_results(_tables["refs"][0], _tables["vqe"][0])

## 7. Decisiones, qué no funcionó y limitaciones

Esta sección junta lo que documentó cada sección (el detalle, con tablas y pruebas, está en el markdown de decisiones de cada una; acá va el resumen y el enlace) y analiza hasta dónde funciona el método, dónde falla y por qué. Las decisiones de infraestructura también están como ADR en `docs/adr/` del repo.

### 7.1 Decisiones

| Etapa | Decisión | Por qué | Detalle |
|---|---|---|---|
| Referencias (#1) | RHF (no UHF) con chequeo de estabilidad interna, CCSD y FCI singlete con PySCF sobre la grilla `D` → `refs`. | Queremos un singlete con orbitales restringidos, como el Hamiltoniano de qubits; HF/CCSD/FCI solo sirven para comparar y elegir puntos de partida (regla del desafío). | Sección 1 |
| Hamiltoniano (#2) | `qubit_hamiltonian(d) -> (op, e_nuc)`; Jordan-Wigner escrito a mano, genérico; orbitales RHF con signo fijo; caché que devuelve copias. | Sin dependencias nuevas y con control exacto del orden de qubits; el signo fijo hace que los coeficientes de $\hat H$ varíen suave con $d$ (si no, el *warm start* arranca lejos); RHF en toda la grilla para que la referencia siga siendo un singlete y $\langle S^2\rangle$ sirva de diagnóstico. | Sección 2, *Decisiones (#2)*; [ADR-0001](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0001-firma-qubit-hamiltonian.md), [ADR-0002](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0002-jordan-wigner-a-mano.md), [ADR-0003](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0003-signo-orbitales-y-cache.md) |
| Ansatz (#4) | 2-QUpCCGSD podado: 19 excitaciones de qubit (pares generalizados + simples con el mismo ángulo en α y β) en 2 capas, como rotaciones de Givens; **465 `cz`**, 19 parámetros. | Llega a FCI en toda la grilla (≤ 0,03 mHa con statevector), donde UCCSD se traba (3,3 mHa a 1,8 Å) por la misma razón que CCSD; conserva $N$ y $S_z$; Givens con multiplexor Gray en vez de `PauliEvolutionGate` (837 → 590 `cz`) y poda de las excitaciones de menor ángulo (590 → 465). | Sección 3, *Decisión (#4)* |
| Optimización (#5) | BFGS con gradiente adjunto exacto en el sector de 36 estados; *warm start* en las dos direcciones con varios arranques por punto y rondas de pulido; se minimiza $E + 0{,}05\,\langle S^2\rangle$. | La regla de *parameter-shift* de 2 términos no vale para excitaciones dobles; los mínimos locales obligan a multi-start; la penalidad evita elegir estados con triplete casi degenerados en energía. Error ≤ 0,003 mHa en toda la grilla. | Sección 4 |
| Medición (#6) | *Pendiente:* `SamplerV2`, agrupamiento de Paulis, reparto de shots, cálculo y validación de $\sigma$. | | Sección 4 |
| Resultados (#7) | *Pendiente:* ajuste de $R_e$, incertidumbre de $R_e$ y $D_e$. | | Sección 5 |
| Confiabilidad (#8) | *Pendiente:* evidencias sin FCI ($\langle S^2\rangle$, varianza/Temple, arranques múltiples, $\sigma$ vs. statevector). | | Sección 6 |

### 7.2 Qué no funcionó

- **`A.compose(B)` para multiplicar operadores (#2).** En Qiskit es $B \cdot A$: daba el Hamiltoniano con partículas y huecos intercambiados, que a medio llenado tiene *el mismo espectro* en el sector $N = 4$. La prueba contra FCI pasaba igual; solo la prueba de HF lo detectó. Lección: hacen falta las dos pruebas. Se usa `@`.
- **Orbitales sin signo fijo (#2, #4).** Entre 76 y 84 términos de Pauli cambiaban de signo de golpe entre distancias vecinas; con un Hamiltoniano provisorio así, el *warm start* del ansatz caía en otros mínimos y aparecía contaminación de triplete ($\langle S^2\rangle$ hasta 3e-2) desde 2,0 Å.
- **UCCSD y 1-UpCCGSD (#4):** no llegan a precisión química en la zona estirada (3,3 y 11 mHa a 1,8 Å).
- **Hardware-efficient (#4):** 42–56 `cz`, pero optimizando la energía queda a 23 mHa de FCI a 1,8 Å; puede representar el estado (optimizando contra el vector FCI, cosa imposible en un VQE real), así que el problema es el paisaje de optimización. Además no conserva $N$ ni $S_z$.
- **QEB-ADAPT-VQE (#4):** recomendado por el análisis previo; llega a FCI pero necesita 461–573 `cz` en línea (sus dobles tocan qubits lejanos) y un circuito distinto por distancia. No mejora al elegido.
- **`EstimatorV2` de `qiskit_aer` para reportar:** no muestrea; devuelve el valor exacto más ruido gaussiano opcional. Equivale a leer el statevector, así que se descartó para los números reportados (#6).
- **Elegir el mejor arranque solo por energía (#5, #16).** En la zona estirada a veces ganaba un estado con $\langle S^2\rangle \approx 3\cdot 10^{-2}$ a solo 0,02 mHa de FCI, y cuál ganaba cambiaba entre corridas por diferencias numéricas de ~1e-13 Ha. Se agregó la penalidad en $\langle S^2\rangle$.
- **Arrancar desde HF o al azar (#5).** Desde $\theta = 0$ el optimizador queda a 150–290 mHa en la zona estirada; con arranques al azar la mediana queda a 0,1–3 mHa. Hace falta el *warm start*; un solo sentido del barrido deja hasta 0,1 mHa, los dos más el pulido bajan a 0,003 mHa.
- *Pendiente (#6–#8):* lo que no funcionó en medición, ajuste de $R_e$ y confiabilidad.

### 7.3 Limitaciones: hasta dónde funciona y dónde falla

**Dónde funciona.** Sin ruido, en este sistema, el ansatz representa el fundamental en toda la grilla (≤ 0,03 mHa con statevector, singlete con $\langle S^2\rangle \leq 3\cdot 10^{-7}$) y el error que queda en la energía reportada es estadístico: lo fija el presupuesto de shots, no el método. Ganarle a CCSD en la zona estirada es holgado (el margen más chico es 9,1 mHa a 1,8 Å); lo exigente es la precisión química, que depende de $\sigma$.

**Dónde falla o deja de valer, y por qué:**

- **Base mínima (STO-3G).** La curva es la del *modelo* $\mathrm{H_4}$/STO-3G: "exacto" quiere decir FCI en esta base, no el $\mathrm{H_4}$ real. $R_e$ y $D_e$ cambian bastante en una base más grande, que además necesitaría muchos más qubits. Y a 3,0 Å la cadena todavía no está del todo disociada: FCI queda ~1 mHa por debajo de cuatro átomos de H aislados, así que $D_e$ (definida por el enunciado con 3,0 Å) está levemente subestimada respecto del límite de disociación.
- **Solo 8 qubits.** El sector relevante ($N = 4$, $S_z = 0$) tiene 36 determinantes: FCI es trivial clásicamente y por eso podemos validar todo contra él. El método no demuestra ventaja cuántica; demuestra el procedimiento. Al crecer el sistema crecen la cantidad de excitaciones (y de `cz`), los términos del Hamiltoniano y los shots necesarios para un $\sigma$ dado.
- **Búsqueda de parámetros con statevector.** Está permitida, pero cuesta exponencial en el número de qubits y aprovecha que el ansatz vive en el sector de 36 estados. En hardware real habría que optimizar con energías y gradientes medidos (con shots y ruido), mucho más lento e inestable.
- **El ansatz se eligió y se podó mirando FCI.** Comparar contra FCI está permitido, pero en un sistema donde FCI no se puede calcular la poda tendría que guiarse por otro criterio (gradientes al estilo ADAPT, varianza de la energía). Por eso la sección 6 arma el argumento de confiabilidad sin FCI.
- **Mínimos locales.** El paisaje tiene mínimos locales: con un solo arranque algunas distancias quedan a 1 mHa o a decenas de mHa de FCI. Usamos varios arranques y *warm start*, pero no hay garantía de mínimo global; la curva suave y los arranques coincidentes son evidencia, no prueba.
- **Contaminación de espín.** El ansatz conserva $N$ y $S_z$ pero no $S^2$ (las excitaciones de qubit no están adaptadas al espín). En la zona estirada el triplete está casi degenerado (0,46 mHa a 3,0 Å), así que una mezcla singlete-triplete casi no cambia la energía ni la varianza: solo $\langle S^2\rangle$ la delata. Hay que medirlo en cada punto juzgado (sección 6).
- **Hartree-Fock deja de ser buena referencia.** RHF es inestable hacia UHF desde ~1,1 Å y su peso en el estado exacto cae a 0,27 a 3,0 Å. Mantenemos orbitales RHF (solo definen la base), pero el estado inicial `HF_BITSTRING` está lejos del fundamental estirado y el ansatz tiene que hacer todo el trabajo multirreferencia.
- **Costo de medición.** $\sigma \propto 1/\sqrt{N_\text{shots}}$: con grupos QWC y reparto óptimo, $\sigma = 1$ mHa cuesta ~2 × 10⁶ shots por punto (`docs/analysis/`). Bajar $\sigma$ a la mitad cuesta 4 veces más shots.
- **Ruido: con 465 `cz` el circuito no sobrevive en hardware sin mitigación.** Con 0,1 % de error por `cz` (modelo de `ruido.py`), la probabilidad de al menos una falla por shot es $1 - 0{,}999^{465} \approx 37\,\%$, sin contar las compuertas de un qubit ni el *readout*. En `FakeFez` circuitos de este tamaño (520–640 `cz`) quedan 600–800 mHa lejos de FCI, peor que Hartree-Fock (`docs/analysis/state-of-the-art.md`, sección 5). Los resultados del desafío central son **sin ruido**; con ruido harían falta un ansatz más corto (la versión de 17 excitaciones tiene 411 `cz`), menos qubits por simetrías y mitigación (extras).
- *Pendiente (#6–#8):* limitaciones que aparezcan en optimización, medición y confiabilidad (p. ej. puntos donde $\sigma$ no alcance para precisión química, o donde $\langle S^2\rangle$ se aleje de 0).

## Extras

Nada de esto entra en el desafío central (ver `docs/challenge/05-extras.md`).

### VQE vs. SQD (#16)

**Fuera del desafío central.** SQD (*Sample-based Quantum Diagonalization*, también QSCI) usa el circuito solo para **muestrear determinantes**; después proyecta el Hamiltoniano en el subespacio que generan esas muestras y lo diagonaliza en forma clásica. El enunciado lo deja afuera del desafío porque en $\mathrm{H_4}$/STO-3G el sector $N = 4$, $S_z = 0$ tiene solo **36 determinantes** (6 cadenas α × 6 cadenas β): con un puñado de shots de cualquier circuito, incluso al azar, se cubre todo y SQD *es* FCI. Lo interesante, entonces, es qué pasa cuando se pone un **tope $k$** al tamaño del subespacio: ¿las muestras cuánticas eligen mejores determinantes que el azar?

Cómo lo hicimos:

- **Muestras:** `SamplerV2` de `qiskit_aer`, 2000 shots por circuito, sobre el circuito transpilado a `cz, sx, x, rz` en línea. Tres fuentes en $d = 1{,}8$, $2{,}4$ y $3{,}0$ Å: el ansatz de la sección 3 (19 excitaciones, 465 `cz`), el mismo con el ruido de `05-extras` (despolarizante 0,1 % por `cz` y 0,005 % por compuerta de un qubit; `ruido.py` no está en el snapshot, así que lo armamos a mano con esos números) y una versión **recortada** a la primera capa (9 excitaciones, 191 `cz`), que como VQE queda lejos de FCI.
- **Parámetros:** si #5 ya dejó `params[d]`, se usa; si no (stand-in), se optimiza el ansatz con statevector en el sector de 36 estados con 20 arranques, minimizando $E + \lambda\langle S^2\rangle$ con $\lambda = 0{,}1$ Ha (el paisaje tiene mínimos locales y el triplete está casi degenerado, ver sección 3 y abajo). El recortado se optimiza igual, siempre.
- **Postselección y recuperación de configuraciones:** los bitstrings con $N_\alpha \neq 2$ o $N_\beta \neq 2$ (solo aparecen con ruido) se corrigen con una versión simplificada de la recuperación de configuraciones de Robledo-Moreno et al.: con las ocupaciones medias $n_q$ de las muestras buenas, a la mitad α o β que tiene electrones de más se le saca el del orbital menos ocupado, y a la que tiene de menos se le agrega en el más ocupado.
- **Tope $k$:** el subespacio son los $k$ determinantes **más frecuentes** (criterio de QSCI). Se proyecta $H$ (el de #2, `sector_hamiltonian`) en esos $k$ y se toma el menor autovalor.
- **Azar:** $k$ determinantes distintos elegidos uniformemente entre los 36 (es lo mismo que quedarse con los $k$ más frecuentes de bitstrings uniformes con el mismo $N$ y $S_z$), 300 repeticiones.
- **Sin tope, estilo `qiskit-addon-sqd`:** subespacio (cadenas α ∪ cadenas β) × (las mismas), simétrico ante α ↔ β.
- **VQE para comparar:** si #6 ya dejó la tabla `vqe`, se usa; si no (stand-in), se mide $\langle H \rangle$ con `SamplerV2` agrupando los términos de Pauli que conmutan qubit a qubit (46 grupos × 100 000 shots), con los mismos $\theta$.

**Por qué SQD a mano y no `qiskit-addon-sqd`:** el issue pedía agregarlo siguiendo la regla de dependencias, pero **no está en el nixpkgs fijado** en `flake.lock`, así que no se puede fijar la misma versión en `requirements.txt`, en el `%pip` y en nix (`nix flake check` fallaría). Con 36 determinantes no hace falta: la proyección es indexar la matriz de 36 × 36 de la sección 3 y diagonalizar con numpy. Lo que reimplementamos (postselección, recuperación de configuraciones, subespacio α × β) es una versión mínima de lo que hace la librería, no una copia exacta (ella, por ejemplo, recupera configuraciones de forma probabilística e iterativa).

Reutiliza de las secciones anteriores: `sector_hamiltonian`, `SECTOR`, `SectorAnsatz`, `sector_ansatz`, `givens_single`, `givens_double`, `ANSATZ_OPS`, `transpile_line`. Tarda alrededor de un minuto.

In [ ]:
# Extras · VQE vs. SQD (#16): muestreo con SamplerV2 y SQD implementado a mano (sin qiskit-addon-sqd).
from qiskit import ClassicalRegister
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import SamplerV2
from pyscf.fci import spin_op

SQD_D = [1.8, 2.4, 3.0]  # Å, zona estirada
SQD_SHOTS = 2_000        # shots por circuito para SQD
SQD_SEED = 16
SQD_STARTS = 20       # arranques del multi-start (stand-in de params)


def circuit_from_ops(ops, name: str) -> QuantumCircuit:
    """Igual que build_ansatz() pero con una lista de excitaciones arbitraria (para la versión recortada)."""
    th = ParameterVector("θ", len(ops))
    qc = QuantumCircuit(N_QUBITS, name=name)
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for t, (_, kind, p, q) in zip(th, ops):
        if kind == "D":
            givens_double(qc, t, alpha_qubit(p), beta_qubit(p), alpha_qubit(q), beta_qubit(q))
        else:
            givens_single(qc, t, alpha_qubit(p), alpha_qubit(q))
            givens_single(qc, t, beta_qubit(p), beta_qubit(q))
    return qc


# Dos circuitos: el ansatz elegido (19 excitaciones) y uno recortado a la primera capa (9), más barato y lejos de FCI.
OPS_SHALLOW = [op for op in ANSATZ_OPS if op[0] == 0]
SQD_CIRCUITS = {"ansatz (19 exc.)": (ANSATZ_OPS, transpile_line(build_ansatz())),
                "recortado (9 exc.)": (OPS_SHALLOW, transpile_line(circuit_from_ops(OPS_SHALLOW, "QUpCCGSD-9")))}
SQD_SECTOR_ANSATZ = {"ansatz (19 exc.)": sector_ansatz, "recortado (9 exc.)": SectorAnsatz(OPS_SHALLOW)}

# Ruido de 05-extras (ruido.py no está en el snapshot): despolarizante 0,1 % por cz y 0,005 % por compuerta de 1 qubit.
NOISE = NoiseModel(basis_gates=BASIS_GATES)
NOISE.add_all_qubit_quantum_error(depolarizing_error(1e-3, 2), ["cz"])
NOISE.add_all_qubit_quantum_error(depolarizing_error(5e-5, 1), ["sx", "x", "rz"])


# S² en el sector de 36 estados (columna j = S²|det_j⟩), para penalizar la mezcla con el triplete.
S2_SECTOR = np.array([spin_op.contract_ss(e.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA)).ravel()
                      for e in np.eye(len(SECTOR))]).T
SPIN_PENALTY = 0.1  # Ha por unidad de ⟨S²⟩


def theta_for(label: str, d: float) -> np.ndarray:
    """θ*: usa `params[d]` de #5 si existe (solo para el ansatz completo); si no, multi-start con statevector
    minimizando E + λ⟨S²⟩ (sin la penalización, en 3,0 Å el mejor arranque salía con ⟨S²⟩ = 0,08)."""
    table = globals().get("params")
    if label == "ansatz (19 exc.)" and isinstance(table, dict):
        for key, x in table.items():
            if abs(float(key) - d) < 1e-6 and len(x) == ansatz.num_parameters:
                return np.asarray(x, dtype=float)
    sa, (H, _) = SQD_SECTOR_ANSATZ[label], sector_hamiltonian(d)
    H_pen = H + SPIN_PENALTY * S2_SECTOR
    rng = np.random.default_rng(SQD_SEED)
    starts = [np.zeros(sa.n)] + [rng.normal(scale=0.7, size=sa.n) for _ in range(SQD_STARTS - 1)]
    return min((minimize(sa.energy_grad, s, args=(H_pen,), jac=True, method="BFGS", options={"gtol": 1e-10})
                for s in starts), key=lambda r: r.fun).x


def measured(qc_t: QuantumCircuit, x: np.ndarray, basis: dict | None = None) -> QuantumCircuit:
    """Asigna θ y mide el qubit lógico v en el clbit v. Ojo: el circuito transpilado permuta qubits al final
    (final_layout); medir con measure_all() sobre el transpilado mezcla α y β (nos pasó: errores de 800 mHa)."""
    qc = qc_t.assign_parameters(x)
    phys = qc_t.layout.final_index_layout()
    for v, b in (basis or {}).items():  # rotación de base para medir X o Y (la usa el VQE de la tabla)
        if b == "Y":
            qc.sdg(phys[v])
        qc.h(phys[v])
    cr = ClassicalRegister(N_QUBITS, "meas")
    qc.add_register(cr)
    for v in range(N_QUBITS):
        qc.measure(phys[v], cr[v])
    return qc


def sample_bitstrings(qc_t, x, shots=SQD_SHOTS, noisy=False) -> np.ndarray:
    """Bitstrings (enteros, q0 = bit menos significativo) en el orden en que salieron, con SamplerV2 de Aer."""
    opts = {"backend_options": {"noise_model": NOISE}} if noisy else {}
    res = SamplerV2(seed=SQD_SEED, options=opts).run([measured(qc_t, x)], shots=shots).result()
    return np.array([int(b, 2) for b in res[0].data.meas.get_bitstrings()])


# --- SQD a mano -----------------------------------------------------------------------------------
# Un determinante es un entero de 8 bits: α = bits 0..3, β = bits 4..7 (convención 0.1). SECTOR (sección 3)
# lista los 36 con N_α = N_β = 2, en el mismo orden que la matriz sector_hamiltonian(d).
POS = {int(s): i for i, s in enumerate(SECTOR)}
_POP = np.array([bin(m).count("1") for m in range(16)])


def in_sector(b: int) -> bool:
    return _POP[b & 15] == N_ALPHA and _POP[b >> N_ORB] == N_BETA


def recover_configurations(shots: np.ndarray) -> tuple[np.ndarray, int]:
    """Postselección + recuperación de configuraciones simplificada (una pasada, determinista).
    Las ocupaciones n_q se estiman con las muestras que ya están en el sector; a cada mitad (α o β) con el
    número equivocado de electrones se le saca el electrón del orbital menos ocupado o se le agrega al más ocupado."""
    good = np.array([in_sector(b) for b in shots])
    occ = np.array([(shots[good] >> q) & 1 for q in range(N_QUBITS)]).mean(axis=1)
    fixed = shots.copy()
    for i in np.flatnonzero(~good):
        out = 0
        for half, n_el in ((0, N_ALPHA), (1, N_BETA)):
            n = occ[N_ORB * half:N_ORB * (half + 1)]
            bits = [(int(shots[i]) >> (N_ORB * half + p)) & 1 for p in range(N_ORB)]
            while sum(bits) > n_el:
                bits[min((p for p in range(N_ORB) if bits[p]), key=lambda p: n[p])] = 0
            while sum(bits) < n_el:
                bits[max((p for p in range(N_ORB) if not bits[p]), key=lambda p: n[p])] = 1
            out |= sum(bt << p for p, bt in enumerate(bits)) << (N_ORB * half)
        fixed[i] = out
    return fixed, int((~good).sum())


def subspace_ground(H: np.ndarray, dets) -> tuple[float, np.ndarray]:
    """Proyecta H en el subespacio de determinantes `dets` y diagonaliza. Devuelve (E, vector en los 36)."""
    idx = [POS[int(b)] for b in dets]
    w, v = np.linalg.eigh(H[np.ix_(idx, idx)])
    psi = np.zeros(len(SECTOR))
    psi[idx] = v[:, 0]
    return w[0], psi


def ranked_dets(shots: np.ndarray) -> list[int]:
    """Determinantes distintos ordenados por frecuencia (el tope k se queda con los k más muestreados)."""
    vals, counts = np.unique(shots, return_counts=True)
    return [int(b) for b in vals[np.argsort(-counts, kind="stable")]]


def spin_square(psi: np.ndarray) -> float:
    return spin_op.spin_square0(psi.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0]


def product_subspace(shots: np.ndarray) -> list[int]:
    """Subespacio estilo qiskit-addon-sqd: (cadenas α ∪ cadenas β) × (las mismas), simétrico en α ↔ β."""
    strings = sorted(set((shots & 15).tolist()) | set((shots >> N_ORB).tolist()))
    return [a | (b << N_ORB) for a in strings for b in strings]


for label, (ops, qc_t) in SQD_CIRCUITS.items():
    print(f"{label}: {qc_t.count_ops().get('cz', 0)} cz")

In [ ]:
# Experimento clave: error de SQD vs. tope k del subespacio, muestras cuánticas vs. al azar (mismo N y S_z).
import matplotlib.pyplot as plt

N_RANDOM_TRIALS = 300
K = np.arange(1, len(SECTOR) + 1)
rng_sqd = np.random.default_rng(SQD_SEED)

SOURCES = [("ansatz (19 exc.)", False), ("ansatz (19 exc.) + ruido", True), ("recortado (9 exc.)", False)]
sqd_curves, sqd_samples, sqd_theta = {}, {}, {}
for d in SQD_D:
    H, e_fci = sector_hamiltonian(d)
    for name, noisy in SOURCES:
        label = name.replace(" + ruido", "")
        if (label, d) not in sqd_theta:
            sqd_theta[label, d] = theta_for(label, d)
        raw = sample_bitstrings(SQD_CIRCUITS[label][1], sqd_theta[label, d], noisy=noisy)
        shots, n_bad = recover_configurations(raw)
        dets = ranked_dets(shots)
        err = np.array([subspace_ground(H, dets[:k])[0] for k in range(1, len(dets) + 1)]) - e_fci
        sqd_curves[name, d] = err
        sqd_samples[name, d] = {"raw": raw, "shots": shots, "n_bad": n_bad, "dets": dets}
    # Azar: k determinantes distintos uniformes entre los 36 (= los k más frecuentes de bitstrings uniformes).
    rand = np.empty((N_RANDOM_TRIALS, len(K)))
    for t in range(N_RANDOM_TRIALS):
        perm = SECTOR[rng_sqd.permutation(len(SECTOR))]
        rand[t] = [subspace_ground(H, perm[:k])[0] for k in K]
    sqd_curves["azar", d] = rand - e_fci

FLOOR = 1e-3  # mHa: el error 0 (subespacio completo) se dibuja en el piso del eje log
COLORS = {"ansatz (19 exc.)": "#2a78d6", "ansatz (19 exc.) + ruido": "#eb6834", "recortado (9 exc.)": "#1baf7a"}
STYLES = {"ansatz (19 exc.)": "-o", "ansatz (19 exc.) + ruido": "--s", "recortado (9 exc.)": ":^"}
fig, axes = plt.subplots(1, len(SQD_D), figsize=(15, 4.3), sharey=True)
for ax, d in zip(axes, SQD_D):
    r = sqd_curves["azar", d] * 1e3
    ax.fill_between(K, np.maximum(np.percentile(r, 10, axis=0), FLOOR), np.maximum(np.percentile(r, 90, axis=0), FLOOR),
                    color="#9a9890", alpha=0.25, lw=0, label="azar (percentiles 10–90)")
    ax.plot(K, np.maximum(np.median(r, axis=0), FLOOR), color="#6b6a65", lw=2, label="azar (mediana)")
    for name, _ in SOURCES:
        e = sqd_curves[name, d] * 1e3
        ax.plot(np.arange(1, len(e) + 1), np.maximum(e, FLOOR), STYLES[name], color=COLORS[name], lw=2, ms=4, label=name, zorder=3 if "recortado" in name else 2)
    ax.axhline(CHEM_ACC * 1e3, color="#52514e", lw=1, ls="-.", label="precisión química")
    ax.set(yscale="log", xlabel="tope k (determinantes en el subespacio)", title=f"d = {d} Å", xlim=(0.5, 36.5))
    ax.grid(alpha=0.3, which="major")
axes[0].set_ylabel("E_SQD − E_FCI (mHa)")
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle(f"SQD con tope k: muestras cuánticas ({SQD_SHOTS} shots, SamplerV2) vs. determinantes al azar (N = 4, S_z = 0)")
fig.tight_layout()
plt.show()


def k_chem(err: np.ndarray) -> float:
    """Menor k con error < precisión química (inf si no se llega con los determinantes muestreados)."""
    ok = np.flatnonzero(err < CHEM_ACC)
    return ok[0] + 1 if len(ok) else np.inf


rows = []
for d in SQD_D:
    row = {"d": d}
    for name, _ in SOURCES:
        s = sqd_samples[name, d]
        row[f"k_quím {name}"] = k_chem(sqd_curves[name, d])
        row[f"dets {name}"] = len(s["dets"])
    kr = [k_chem(e) for e in sqd_curves["azar", d]]
    row["k_quím azar (mediana)"], row["k_quím azar (p90)"] = np.median(kr), np.percentile(kr, 90)
    row["fuera del sector (ruido)"] = f"{sqd_samples['ansatz (19 exc.) + ruido', d]['n_bad']}/{SQD_SHOTS}"
    # Un subespacio de k determinantes sueltos no es cerrado bajo S²: el autovector puede mezclar singlete y triplete.
    H, _ = sector_hamiltonian(d)
    smp = sqd_samples["ansatz (19 exc.)", d]
    row["⟨S²⟩ ansatz, k = 10"] = round(spin_square(subspace_ground(H, smp["dets"][:10])[1]), 4)
    row["⟨S²⟩ ansatz, α×β"] = round(spin_square(subspace_ground(H, product_subspace(smp["shots"]))[1]), 4)
    rows.append(row)
sqd_kchem = pd.DataFrame(rows).set_index("d")
sqd_kchem.T

In [ ]:
# Tabla SQD vs. VQE en la zona estirada: error, shots y cz.
# VQE: si #6 ya dejó la tabla `vqe`, se usa; si no (stand-in), se mide ⟨H⟩ con SamplerV2 agrupando los Pauli
# que conmutan qubit a qubit (QWC), con los mismos θ que usa SQD.
VQE_SHOTS_PER_GROUP = 100_000
_PARITY8 = np.array([bin(m).count("1") & 1 for m in range(2 ** N_QUBITS)])


def vqe_sampled_energy(qc_t, x, d, shots=VQE_SHOTS_PER_GROUP) -> tuple[float, float, int]:
    op, e_nuc = qubit_hamiltonian(d)
    groups = op.group_commuting(qubit_wise=True)
    pubs, terms = [], []
    for g in groups:
        px, pz = g.paulis.x, g.paulis.z  # (términos, qubits); qubit v = columna v
        basis = {v: ("Y" if (px[:, v] & pz[:, v]).any() else "X") for v in range(N_QUBITS) if px[:, v].any()}
        masks = ((px | pz) * (1 << np.arange(N_QUBITS))).sum(axis=1)
        pubs.append(measured(qc_t, x, basis))
        terms.append((masks, g.coeffs.real))
    res = SamplerV2(seed=SQD_SEED).run(pubs, shots=shots).result()
    energy, var = e_nuc, 0.0
    for r, (masks, coeffs) in zip(res, terms):
        b = np.array([int(s, 2) for s in r.data.meas.get_bitstrings()])
        f = coeffs @ (1 - 2 * _PARITY8[b[None, :] & masks[:, None]])  # estimador por shot del grupo
        energy += f.mean()
        var += f.var(ddof=1) / shots
    return energy, np.sqrt(var), shots * len(groups)


def first_full_product(shots: np.ndarray) -> int:
    """Shots hasta que el subespacio α×β cubre los 36 determinantes (ahí SQD sin tope = FCI exacto)."""
    for n in range(1, len(shots) + 1):
        if len(product_subspace(shots[:n])) == len(SECTOR):
            return n
    return -1


cz_of = {name: SQD_CIRCUITS[name.replace(" + ruido", "")][1].count_ops()["cz"] for name, _ in SOURCES}
cz_of["azar"] = 0
vqe_table = globals().get("vqe")
vs_rows = {}


def put(method, d, cz, shots, dim, err, sigma=None):
    r = vs_rows.setdefault(method, {"método": method, "cz": cz, "shots": [], "dim": []})
    r["shots"].append(shots)
    r["dim"].append(dim)
    r[f"error {d} Å (mHa)"] = f"{round(err * 1e3, 3) + 0.0:.3f}" + ("" if sigma is None else f" ± {sigma * 1e3:.2f}")


def _join(vals):  # "2000" si es igual en las tres distancias, "31 / 29 / 27" si no
    return str(vals[0]) if len(set(vals)) == 1 else " / ".join(map(str, vals))


for d in SQD_D:
    H, e_fci = sector_hamiltonian(d)
    if isinstance(vqe_table, pd.DataFrame) and np.isclose(vqe_table.d, d).any():
        v = vqe_table[np.isclose(vqe_table.d, d)].iloc[0]
        e, s, n, cz, src = v.energy, v.sigma, int(v.shots), int(v.cz), "VQE (#6)"
    else:
        e, s, n = vqe_sampled_energy(SQD_CIRCUITS["ansatz (19 exc.)"][1], sqd_theta["ansatz (19 exc.)", d], d)
        cz, src = cz_of["ansatz (19 exc.)"], "VQE (stand-in: QWC con SamplerV2)"
    put(src, d, cz, n, "–", e - e_fci, sigma=s)
    for name, _ in SOURCES:
        smp = sqd_samples[name, d]
        put(f"SQD {name}, tope k = 10", d, cz_of[name], SQD_SHOTS, 10, sqd_curves[name, d][9])
        put(f"SQD {name}, sin tope", d, cz_of[name], SQD_SHOTS, len(smp["dets"]), sqd_curves[name, d][-1])
    put("SQD azar, tope k = 10 (mediana de 300)", d, 0, "–", 10, np.median(sqd_curves["azar", d][:, 9]))
    # α×β: con el primer puñado de shots ya aparecen las 6 cadenas y el subespacio es todo el sector (= FCI).
    for name in ("ansatz (19 exc.)", "recortado (9 exc.)", "azar"):
        shots = sqd_samples[name, d]["shots"] if name != "azar" else SECTOR[rng_sqd.integers(len(SECTOR), size=SQD_SHOTS)]
        n = first_full_product(shots)
        put(f"SQD {name}, α×β (shots hasta cubrir el sector)", d, cz_of[name], n, len(SECTOR),
            subspace_ground(H, product_subspace(shots[:n]))[0] - e_fci)
for r in vs_rows.values():
    r["shots"], r["dim"] = _join(r["shots"]), _join(r["dim"])
sqd_vs_vqe = pd.DataFrame(vs_rows.values()).set_index("método")
sqd_vs_vqe

#### Resultados y decisiones (#16)

**Error vs. tope $k$ (gráfico).** Con el subespacio limitado, las muestras cuánticas le ganan claramente al azar:

| | 1,8 Å | 2,4 Å | 3,0 Å |
|---|---|---|---|
| $k$ mínimo para precisión química, ansatz (19 exc.) | 16 | 12 | 6 |
| ídem, ansatz + ruido | 24 | 30 | 6 |
| ídem, recortado (9 exc.) | no llega | no llega | 7 |
| ídem, azar (mediana / percentil 90) | 35 / 36 | 34 / 36 | 28 / 30 |
| error con $k = 10$, ansatz (mHa) | 5,1 | 2,3 | 0,39 |
| error con $k = 10$, azar, mediana (mHa) | 318 | 318 | 332 |

- **Por qué gana lo cuántico:** el fundamental FCI tiene solo ~16 determinantes con peso apreciable (los otros 20 son cero por la simetría espacial de la cadena). El ansatz los muestrea con frecuencia proporcional a $|c_i|^2$, así que los $k$ más frecuentes son casi los $k$ más importantes. El azar gasta más de la mitad del tope en determinantes de peso cero y necesita casi todo el sector para llegar a precisión química.
- **El ruido hace las dos cosas.** Con ~27 % de shots fuera del sector, la recuperación de configuraciones mete determinantes poco relevantes entre los más frecuentes: con tope, empeora (a 2,4 Å necesita $k = 30$ en lugar de 12). Sin tope, esos mismos determinantes "basura" terminan cubriendo los 36 y el resultado es exacto: el ruido *ayuda* a SQD sin tope, que es justamente la trampa del enunciado.
- **El recortado** muestrea solo los determinantes que puede alcanzar y a 1,8 y 2,4 Å se estanca por encima de precisión química. Igual mejora mucho sobre su propio VQE: con statevector, ese circuito queda a 19,9 / 8,9 / 1,0 mHa de FCI; SQD con sus muestras baja a 4,9 / 2,3 / 0,25 mHa.
- **Espín:** un subespacio de $k$ determinantes sueltos no es cerrado ante $S^2$, y el autovector mezcla singlete con triplete: con $k = 10$, $\langle S^2\rangle$ = 0,13 / 0,44 / 0,60. El subespacio α × β (simétrico) da $\langle S^2\rangle = 0$. En la zona estirada el triplete está casi degenerado, así que la energía sola no lo delata (igual que en la sección 3).

**SQD vs. VQE (tabla).** Sin tope, SQD con el ansatz llega a 0,60 / 0,13 / 0,09 mHa de FCI con 2000 shots (16–18 determinantes distintos), y con el subespacio α × β es exacto con **9–31 shots** del ansatz o **6–8 bitstrings al azar** (lo cuántico tarda igual o más, porque concentra las muestras en pocos determinantes). El VQE stand-in, con 4,6 millones de shots, queda en 1,0 ± 1,2 / 0,3 ± 1,1 / 0,04 ± 1,1 mHa: el costo de VQE es la medición, el de SQD es la diagonalización clásica, que acá es trivial (36 × 36) y en sistemas grandes es el cuello de botella. Los dos usan el mismo circuito (465 `cz`); sin tope, SQD también llega a FCI con el recortado (191 `cz`). Por eso este resultado **no** dice que SQD sea mejor para $\mathrm{H_4}$: dice que $\mathrm{H_4}$ es demasiado chico para distinguir un buen muestreo de uno malo salvo que se limite el subespacio.

Qué no funcionó o descartamos:

- **`qiskit-addon-sqd`**: no está en el nixpkgs fijado (ver arriba).
- **Medir después de transpilar con `measure_all()`**: el transpilador permuta los qubits al final (`final_layout`), así que los bits medidos no correspondían a la convención 0.1 y el determinante más frecuente "no era" HF (errores de ~800 mHa con $k = 1$). Se mide cada qubit lógico en su qubit físico con `layout.final_index_layout()`.
- **Multi-start eligiendo solo por energía**: en 2,4 y 3,0 Å el mínimo elegido cambiaba según qué celdas se hubieran corrido antes (el Hamiltoniano difería en ~1e-13 Ha por la convergencia de RHF, y eso alcanza para que BFGS caiga en otro mínimo de un paisaje casi plano). En 3,0 Å el mejor arranque tenía $\langle S^2\rangle = 0{,}08$ y, en una corrida, el recortado convergió a un estado casi triplete ($\langle S^2\rangle = 1{,}98$). Con la penalización $\lambda\langle S^2\rangle$ y 20 arranques el resultado es el mismo en las dos situaciones y todos los estados son singlete.
- **Tope sobre el subespacio α × β** (quedarse con las $m$ cadenas más frecuentes, dimensión $m^2$): lo descartamos de entrada, porque con solo 6 cadenas la curva tendría 6 puntos (dimensión 1, 4, 9, 16, 25, 36) y a $m = 6$ ya es FCI.

Limitaciones:

- Los $\theta$ del ansatz son los `params` de #5 (sección 4). El VQE de la tabla es un stand-in (QWC, sin optimizar la cantidad de shots) hasta que llegue `vqe` de #6. El código toma los reales apenas existan.
- El ruido es el modelo de `05-extras` armado a mano, sin errores de lectura ni conectividad de `FakeFez`.
- Los números dependen de la semilla (`SQD_SEED`); el $k$ del azar se reporta con mediana y percentil 90 de 300 sorteos.

### Otra geometría (#17)

Mismo pipeline, cambiando solo la geometría: **cuadrado de lado $d$** (plano xy) frente a la cadena lineal, en una grilla reducida (11 distancias). El cuadrado es el caso multirreferencia por excelencia: con simetría $D_{4h}$ los dos orbitales del medio son degenerados, así que con 4 electrones el determinante de HF está mal definido en todas las distancias, no solo en la zona estirada.

- `h4_geometry_shape(d, shape)` agrega la forma **sin tocar el contrato 0.1**: con `shape="linear"` devuelve exactamente `h4_geometry(d)`, y para la cadena lineal se reutilizan `h4_scf`/`qubit_hamiltonian` de la sección 2 tal cual.
- Referencias HF/CCSD/FCI con PySCF, Hamiltoniano con la maquinaria de la sección 2, parámetros con el simulador del sector de la sección 3 (statevector: permitido para buscar θ) y **energías reportadas medidas con `SamplerV2`** de `qiskit_aer`, con σ desde los bitstrings.
- Decisiones, resultados y limitaciones al final de la subsección.

In [ ]:
# Extra #17 (1/4): geometría con forma y referencias clásicas (HF, CCSD, FCI) con PySCF.
# No toca el contrato: `h4_geometry_shape(d, "linear")` devuelve exactamente `h4_geometry(d)`, y para la
# cadena lineal se reutilizan `h4_scf` / `qubit_hamiltonian` de la sección 2 tal cual.
from pyscf import cc, lib as pyscf_lib

GEO_SHAPES = ("linear", "square", "rectangle")
GEO_D = np.array([0.8, 0.9, 1.0, 1.2, 1.3, 1.4, 1.6, 1.8, 2.0, 2.4, 3.0])  # Å, grilla reducida común


def h4_geometry_shape(d: float, shape: str = "linear", aspect: float = 1.5) -> str:
    """H4 con forma: cadena lineal (contrato 0.1), cuadrado de lado d o rectángulo d × aspect·d (plano xy)."""
    if shape == "linear":
        return h4_geometry(d)
    if shape == "square":
        aspect = 1.0
    elif shape != "rectangle":
        raise ValueError(f"forma desconocida: {shape!r} (opciones: {GEO_SHAPES})")
    pts = [(0, 0), (d, 0), (d, aspect * d), (0, aspect * d)]
    return "; ".join(f"H {x:.6f} {y:.6f} 0.000000" for x, y in pts)


def _fix_orbital_signs(mf: scf.hf.RHF, d: float) -> scf.hf.RHF:
    """Mismo criterio que la sección 2 (ADR-0003): primer coeficiente con |c| > 1e-4 positivo."""
    c = mf.mo_coeff
    significant = np.abs(c) > 1e-4
    if not significant.any(axis=0).all():
        raise RuntimeError(f"Orbital sin coeficientes > 1e-4 en d = {d}: no se puede fijar el signo")
    first = np.argmax(significant, axis=0)
    mf.mo_coeff = c * np.sign(c[first, np.arange(c.shape[1])])
    return mf


@lru_cache(maxsize=None)
def _scf_shape_cached(d: float, shape: str, aspect: float) -> scf.hf.RHF:
    if shape == "linear":
        return _h4_scf_cached(d)  # el RHF de la sección 2, sin cambios
    mol = gto.M(atom=h4_geometry_shape(d, shape, aspect), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    # En el cuadrado el RHF cae en soluciones distintas según d (no todas mínimas): seguimos la
    # inestabilidad interna (RHF → RHF) hasta un mínimo, así todas las distancias usan la misma clase de solución.
    for _ in range(10):
        mo, _, stable, _ = mf.stability(return_status=True)
        if stable:
            break
        mf.kernel(mf.make_rdm1(mo, mf.mo_occ))
    if not (mf.converged and stable):
        raise RuntimeError(f"RHF no llegó a un mínimo estable en d = {d}, forma {shape}")
    return _fix_orbital_signs(mf, d)


def scf_shape(d: float, shape: str = "linear", aspect: float = 1.5) -> scf.hf.RHF:
    mf = copy.copy(_scf_shape_cached(_distance_key(d), shape, float(aspect)))
    for attr in ("mo_coeff", "mo_energy", "mo_occ"):
        setattr(mf, attr, getattr(mf, attr).copy())
    return mf


def references_shape(d: float, shape: str = "linear", aspect: float = 1.5) -> dict:
    """Como `references` (#1) pero para cualquier forma; agrega diagnósticos de CCSD y del espín."""
    mf = scf_shape(d, shape, aspect)
    mycc = cc.CCSD(mf)
    mycc.max_cycle = 100
    mycc.kernel()
    solver = fci.FCI(mf)
    solver.nroots = 2
    e, ci = solver.kernel()
    s2 = [spin_op.spin_square0(v, N_ORB, (N_ALPHA, N_BETA))[0] for v in ci]
    return {"shape": shape, "d": float(d), "e_hf": mf.e_tot, "e_ccsd": mycc.e_tot, "ccsd_conv": mycc.converged,
            "e_fci": e[0], "S2_fci": s2[0], "gap_mHa": (e[1] - e[0]) * 1e3, "S2_exc": s2[1],
            "t2_max": np.abs(mycc.t2).max(), "err_ccsd_mHa": (mycc.e_tot - e[0]) * 1e3}


# Con matrices de 4×4 los hilos de OpenMP de PySCF solo agregan overhead (~10× más lento): un hilo acá.
with pyscf_lib.with_omp_threads(1):
    geo_refs = pd.DataFrame([references_shape(d, s) for s in ("linear", "square") for d in GEO_D])
    geo_rect = pd.DataFrame([references_shape(1.2, "rectangle", a) | {"aspect": a}
                             for a in (1.0, 1.05, 1.1, 1.2, 1.4, 1.7, 2.0)])
print("Error de CCSD vs. FCI (mHa); negativo = CCSD por debajo de FCI (no variacional):")
print(geo_refs.pivot(index="d", columns="shape", values="err_ccsd_mHa").round(2).to_string())
print("\nCCSD sin converger:", geo_refs.loc[~geo_refs.ccsd_conv, ["shape", "d"]].values.tolist())
print("Cuadrado: estado fundamental de FCI (S_z = 0) con ⟨S²⟩ ≈ 2 (triplete) en d =",
      geo_refs.query("shape == 'square' and S2_fci > 1").d.tolist())

# Rectángulo d × a·d a d = 1,2 Å: de cuadrado (a = 1) a dos H2 separados. Muestra que el problema de CCSD
# viene de la casi degeneración del cuadrado, no del plano.
print("\nRectángulo a d = 1,2 Å:")
print(geo_rect[["aspect", "e_fci", "gap_mHa", "t2_max", "err_ccsd_mHa"]].round(4).to_string(index=False))
geo_refs.round(6)

In [ ]:
# Extra #17 (2/4): Hamiltoniano de qubits y búsqueda de parámetros (statevector, permitido) para el cuadrado.
# Misma maquinaria de la sección 2 (integrales en base MO → `spatial_to_spin_orbitals` → `jordan_wigner`)
# y el mismo simulador en el sector de 36 estados de la sección 3 (`SectorAnsatz`).
@lru_cache(maxsize=None)
def _qubit_hamiltonian_shape_cached(d: float, shape: str, aspect: float) -> tuple[SparsePauliOp, float]:
    if shape == "linear":
        return _qubit_hamiltonian_cached(d)  # el Hamiltoniano de la sección 2, sin cambios
    mf = _scf_shape_cached(d, shape, aspect)
    c = mf.mo_coeff
    h1 = c.T @ mf.get_hcore() @ c
    eri = ao2mo.restore(1, ao2mo.full(mf.mol, c), c.shape[1])
    return jordan_wigner(*spatial_to_spin_orbitals(h1, eri)), mf.mol.energy_nuc()


def qubit_hamiltonian_shape(d: float, shape: str = "linear", aspect: float = 1.5) -> tuple[SparsePauliOp, float]:
    """Como `qubit_hamiltonian(d)` (#2) para cualquier forma: (H electrónico de 8 qubits, e_nuc)."""
    op, e_nuc = _qubit_hamiltonian_shape_cached(_distance_key(d), shape, float(aspect))
    return op.copy(), e_nuc


def sector_hamiltonian_shape(d: float, shape: str) -> tuple[np.ndarray, float]:
    op, e_nuc = qubit_hamiltonian_shape(d, shape)
    M = op.to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    H = M.real + e_nuc * np.eye(len(SECTOR))
    return H, np.linalg.eigvalsh(H)[0]


# Control: el Hamiltoniano de 8 qubits reproduce HF y FCI de PySCF también en el cuadrado.
for _d in (1.3, 3.0):
    _op, _en = qubit_hamiltonian_shape(_d, "square")
    _r = geo_refs.query("shape == 'square' and d == @_d").iloc[0]
    assert abs(hf_energy_from_op(_op) + _en - _r.e_hf) < 1e-8 and abs(sector_ground_energy(_op) + _en - _r.e_fci) < 1e-8
assert np.allclose(qubit_hamiltonian_shape(1.8)[0].coeffs, qubit_hamiltonian(1.8)[0].coeffs)  # lineal = sección 2

# El ansatz elegido (podado sobre la cadena lineal) y el 2-QUpCCGSD completo (12 excitaciones × 2 capas).
GEO_FULL_OPS = [(l, k, p, q) for l in (0, 1) for k in ("D", "S") for p, q in itertools.combinations(range(N_ORB), 2)]
geo_ansatz = {"podado-19": sector_ansatz if "sector_ansatz" in globals() else SectorAnsatz(ANSATZ_OPS),
              "completo-24": SectorAnsatz(GEO_FULL_OPS)}
GEO_STARTS = 10  # cero, el óptimo de la distancia anterior y 8 al azar (como la sección 3)


def geo_optimize(shape: str, name: str, grid, seed: int = 0) -> pd.DataFrame:
    sa, rng = geo_ansatz[name], np.random.default_rng(seed)
    rows, x_prev = [], np.zeros(sa.n)
    for d in grid:
        H, e_fci = sector_hamiltonian_shape(d, shape)
        starts = [np.zeros(sa.n), x_prev] + [rng.normal(scale=0.7, size=sa.n) for _ in range(GEO_STARTS - 2)]
        runs = [minimize(sa.energy_grad, s, args=(H,), jac=True, method="BFGS", options={"gtol": 1e-8}) for s in starts]
        errs = np.array([r.fun - e_fci for r in runs]) * 1e3
        best = runs[int(np.argmin(errs))]
        x_prev = best.x
        s2 = spin_op.spin_square0(sa.state(best.x).reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0]
        rows.append({"shape": shape, "ansatz": name, "d": float(d), "e_sv": best.fun, "e_fci": e_fci,
                     "err_sv_mHa": errs.min(), "ok_starts": f"{(errs < CHEM_ACC * 1e3).sum()}/{len(runs)}",
                     "S2": s2, "theta": best.x})
    return pd.DataFrame(rows)


GEO_VQE_D = {"linear": GEO_D, "square": GEO_D[GEO_D >= 0.9]}  # cuadrado: en 0,8 Å el fundamental es triplete
geo_opt = pd.concat([geo_optimize("linear", "podado-19", GEO_VQE_D["linear"]),
                     geo_optimize("square", "podado-19", GEO_VQE_D["square"]),
                     geo_optimize("square", "completo-24", GEO_VQE_D["square"])], ignore_index=True)
print("Statevector (solo para elegir θ; no son energías reportadas). Error vs. FCI en mHa y arranques "
      f"que llegan a precisión química ({CHEM_ACC * 1e3} mHa):")
print(geo_opt.pivot(index="d", columns=["shape", "ansatz"], values=["err_sv_mHa", "ok_starts"]).to_string(
    float_format=lambda v: f"{v:.4f}"))
print(f"⟨S²⟩ máximo: " + ", ".join(f"{s}/{a}: {g.S2.max():.1e}" for (s, a), g in geo_opt.groupby(["shape", "ansatz"])))

In [ ]:
# Extra #17 (3/4): energía MEDIDA con SamplerV2 de qiskit_aer (lo único que se reporta).
# Stand-in compacto de la medición de #6 (todavía no está en main): grupos QWC del SparsePauliOp, una
# rotación de base por grupo, shots por grupo ∝ Σ|c_k| del grupo, y E y σ desde los bitstrings.
from qiskit_aer.primitives import SamplerV2

GEO_SHOTS = 3_000_000  # por punto (suma sobre grupos)


def qwc_measurement_plan(op: SparsePauliOp) -> tuple[float, list]:
    """(término constante, [(base por qubit 'Z'/'X'/'Y', máscaras de bits, coeficientes)] por grupo QWC)."""
    is_id = ~(op.paulis.x.any(axis=1) | op.paulis.z.any(axis=1))
    const = op.coeffs[is_id].real.sum()
    plan = []
    for g in SparsePauliOp(op.paulis[~is_id], op.coeffs[~is_id]).group_commuting(qubit_wise=True):
        x, z = g.paulis.x, g.paulis.z
        basis = np.where((x & z).any(axis=0), "Y", np.where(x.any(axis=0), "X", "Z"))
        masks = ((x | z) * (1 << np.arange(op.num_qubits))).sum(axis=1)
        plan.append((tuple(basis), masks, g.coeffs.real))
    return const, plan


def qwc_sampler_energy(qc: QuantumCircuit, theta: np.ndarray, op: SparsePauliOp, e_nuc: float,
                       shots: int = GEO_SHOTS, seed: int = 0) -> dict:
    const, plan = qwc_measurement_plan(op)
    weights = np.array([np.abs(c).sum() for _, _, c in plan])
    n_shots = np.maximum(np.round(shots * weights / weights.sum()).astype(int), 100)
    energy, var = const + e_nuc, 0.0
    for g, ((basis, masks, coeffs), n) in enumerate(zip(plan, n_shots)):
        circ = qc.copy()
        for q, b in enumerate(basis):
            if b == "Y":
                circ.sdg(q)
            if b != "Z":
                circ.h(q)
        circ.measure_all()
        # Un job por grupo con su propia semilla: en qiskit-aer 0.17.2, PUBs del mismo job con distinta
        # cantidad de shots comparten el flujo aleatorio y sus resultados quedan correlacionados (ver markdown).
        sub_seed = int(np.random.SeedSequence([seed, g]).generate_state(1)[0])
        meas = SamplerV2(seed=sub_seed).run([(circ, theta, int(n))]).result()[0].data.meas
        freq = np.bincount(meas.array[:, -1], minlength=2 ** op.num_qubits).astype(float)  # 8 qubits: 1 byte por shot
        parity = np.array([[(b & m).bit_count() & 1 for m in masks] for b in range(len(freq))])
        vals = (1 - 2 * parity) @ coeffs  # valor del grupo completo en cada bitstring (incluye covarianzas)
        mean = freq @ vals / n
        energy += mean
        var += freq @ (vals - mean) ** 2 / (n - 1) / n
    return {"energy": energy, "sigma": np.sqrt(var), "shots": int(n_shots.sum()), "groups": len(plan)}


geo_circuit = transpile(build_ansatz(), basis_gates=["cx", "u"], optimization_level=1)  # para Aer (más rápido)
geo_cz = cz_count(build_ansatz())
_rows = []
for _k, _r in geo_opt.query("ansatz == 'podado-19'").iterrows():
    _op, _en = qubit_hamiltonian_shape(_r.d, _r["shape"])
    _m = qwc_sampler_energy(geo_circuit, _r.theta, _op, _en, seed=_k)
    _rows.append({"shape": _r["shape"], "d": _r.d, **_m, "cz": geo_cz, "e_sv": _r.e_sv, "e_fci": _r.e_fci})
geo_vqe = pd.DataFrame(_rows)
geo_vqe["err_mHa"] = (geo_vqe.energy - geo_vqe.e_fci) * 1e3
geo_vqe["sigma_mHa"] = geo_vqe.sigma * 1e3
geo_vqe["dev_sv_in_sigma"] = (geo_vqe.energy - geo_vqe.e_sv) / geo_vqe.sigma
print(f"Grupos QWC por punto: {sorted(int(g) for g in geo_vqe.groups.unique())}; shots por punto: {GEO_SHOTS:.0e}; "
      f"medida vs. statevector: máx. {geo_vqe.dev_sv_in_sigma.abs().max():.1f} σ")

# Validación de σ: repetir la medición con otras semillas en un punto del cuadrado y comparar la dispersión.
_r = geo_opt.query("shape == 'square' and ansatz == 'podado-19' and d == 1.8").iloc[0]
_op, _en = qubit_hamiltonian_shape(1.8, "square")
_rep = [qwc_sampler_energy(geo_circuit, _r.theta, _op, _en, shots=200_000, seed=1000 + s) for s in range(20)]
_e = np.array([x["energy"] for x in _rep])
print(f"Validación de σ (cuadrado, 1,8 Å, 2e5 shots, 20 semillas): dispersión {_e.std(ddof=1) * 1e3:.2f} mHa, "
      f"σ calculada {np.mean([x['sigma'] for x in _rep]) * 1e3:.2f} mHa, media − statevector "
      f"{(_e.mean() - _r.e_sv) * 1e3:+.2f} ± {_e.std(ddof=1) / np.sqrt(len(_e)) * 1e3:.2f} mHa")
geo_vqe[["shape", "d", "energy", "sigma_mHa", "shots", "cz", "err_mHa", "dev_sv_in_sigma"]].round(6)

In [ ]:
# Extra #17 (4/4): curva del cuadrado vs. cadena lineal, error vs. d, R_e y D_e.
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline


def re_de(d: np.ndarray, e: np.ndarray) -> tuple[float, float]:
    """R_e = mínimo del spline cúbico de E(d); D_e = E(3,0 Å) − E(R_e) (definición de la celda 0.1)."""
    spline, fine = CubicSpline(d, e), np.linspace(d.min(), d.max(), 4001)
    i = np.argmin(spline(fine))
    return fine[i], spline(DISSOCIATED) - spline(fine[i])


def re_de_sampled(d, e, sigma, n: int = 500, seed: int = 0):
    """R_e, D_e de la curva medida y su incertidumbre (bootstrap paramétrico: E_i ~ N(energy_i, σ_i))."""
    rng = np.random.default_rng(seed)
    boot = np.array([re_de(d, e + sigma * rng.standard_normal(len(e))) for _ in range(n)])
    return re_de(d, e), boot.std(axis=0)


_summary = []
for shape in ("linear", "square"):
    rf = geo_refs.query("shape == @shape")
    vq = geo_vqe.query("shape == @shape")
    rf_v = rf[rf.d.isin(vq.d)]
    (re_v, de_v), (dre, dde) = re_de_sampled(vq.d.values, vq.energy.values, vq.sigma.values)
    re_f, de_f = re_de(rf_v.d.values, rf_v.e_fci.values)
    cc_ok = rf_v[rf_v.ccsd_conv]
    re_c, de_c = re_de(cc_ok.d.values, cc_ok.e_ccsd.values)
    if cc_ok.d.max() < DISSOCIATED:  # sin CCSD convergido a 3,0 Å no hay D_e (el spline extrapolaría)
        de_c = np.nan
    _summary.append({"forma": shape, "R_e FCI (Å)": re_f, "R_e VQE (Å)": f"{re_v:.3f} ± {dre:.3f}",
                    "R_e CCSD (Å)": re_c, "D_e FCI (mHa)": de_f * 1e3, "D_e VQE (mHa)": f"{de_v * 1e3:.1f} ± {dde * 1e3:.1f}",
                    "D_e CCSD (mHa)": de_c * 1e3,
                    "máx |err CCSD| (mHa)": rf_v.err_ccsd_mHa.abs().max(),
                    "máx |err VQE| (mHa)": vq.err_mHa.abs().max(), "máx σ (mHa)": vq.sigma_mHa.max()})
geo_summary = pd.DataFrame(_summary).set_index("forma")
print("CCSD del cuadrado no converge a 3,0 Å: su D_e queda sin definir (NaN) y no se grafica ese punto.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = {"linear": "tab:blue", "square": "tab:red"}
names = {"linear": "lineal", "square": "cuadrado"}
for shape, col in colors.items():
    rf, vq = geo_refs.query("shape == @shape"), geo_vqe.query("shape == @shape")
    rf = rf[rf.d.isin(vq.d)]  # cuadrado sin 0,8 Å: ahí el fundamental del sector es triplete
    ok = rf.ccsd_conv
    axes[0].plot(rf.d, rf.e_fci, "-", color=col, lw=1.2, label=f"FCI ({names[shape]})")
    axes[0].plot(rf.d[ok], rf.e_ccsd[ok], "--", color=col, lw=1, alpha=0.7, label=f"CCSD ({names[shape]})")
    axes[0].errorbar(vq.d, vq.energy, yerr=vq.sigma, fmt="o", ms=4, color=col, capsize=2,
                     label=f"VQE medido ± σ ({names[shape]})")
    axes[1].plot(rf.d[ok], rf.err_ccsd_mHa[ok], "--s", ms=3, color=col, alpha=0.7, label=f"CCSD − FCI ({names[shape]})")
    axes[1].errorbar(vq.d, vq.err_mHa, yerr=vq.sigma_mHa, fmt="o", ms=4, color=col, capsize=2,
                     label=f"VQE − FCI ± σ ({names[shape]})")
axes[0].set(xlabel="d (Å)", ylabel="Energía total (Ha)", title="$\\mathrm{H_4}$ lineal vs. cuadrado (STO-3G)")
axes[1].axhspan(-CHEM_ACC * 1e3, CHEM_ACC * 1e3, color="0.85", label="precisión química")
axes[1].axhline(0, color="k", lw=0.6)
axes[1].set(xlabel="d (Å)", ylabel="Error vs. FCI (mHa)", title="Error de CCSD y de VQE (medido)")
for ax in axes:
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
geo_summary.round(3)

#### Decisiones, resultados y limitaciones (#17)

**Geometría.** `h4_geometry_shape(d, shape, aspect)`: `"linear"` (devuelve `h4_geometry(d)` sin cambios), `"square"` (lado $d$) y `"rectangle"` ($d \times a\,d$). No se tocó la celda 0.1: todo vive en esta subsección, con funciones y tablas propias (`scf_shape`, `qubit_hamiltonian_shape`, `references_shape`, `geo_refs`, `geo_opt`, `geo_vqe`) para no pisar `refs`/`vqe`/`params` del desafío central. La grilla es reducida (11 distancias, 0,8–3,0 Å); el VQE del cuadrado empieza en 0,9 Å porque a 0,8 Å el fundamental del sector $S_z = 0$ es un **triplete** (1,5 mHa por debajo del singlete, regla de Hund por la degeneración) y el desafío es el singlete.

**RHF del cuadrado.** Con simetría $D_{4h}$ el determinante cerrado no existe (dos electrones en un par degenerado), así que RHF rompe la simetría y PySCF cae en soluciones **distintas según $d$**, no todas mínimas: sin más cuidado, $E_\text{HF}$ salta (54 mHa más alta a 0,8 Å y 76 mHa a 1,2 Å que la solución estable) y CCSD sobre esa referencia erra +148 y +180 mHa. Por eso `scf_shape` sigue la inestabilidad interna (`mf.stability()`, RHF → RHF) hasta un mínimo en todas las distancias. Se mantienen orbitales RHF (no UHF), igual que en la sección 2 (ADR-0003). El criterio de signo del ADR-0003 sigue funcionando (todo orbital tiene algún coeficiente > 1e-4, se verifica), pero hay dos soluciones RHF equivalentes (rotadas 90°), así que la continuidad entre distancias no está garantizada: el *warm start* vale menos y el multi-start es lo que manda.

**Dónde falla CCSD** (tabla de arriba, error vs. FCI):

| | lineal | cuadrado |
|---|---|---|
| ¿Desde dónde > precisión química? | 1,6 Å (−3,0 mHa) | **0,9 Å** (−2,6 mHa) |
| Peor error | −30,6 mHa (2,4 Å) | −19,6 mHa (2,0 Å) |
| Signo | por debajo de FCI (no variacional) | por debajo de FCI en todas las distancias convergidas |
| Convergencia | siempre | **no converge a 3,0 Å** (el valor depende de cuántas iteraciones se dejen) |
| $\max\lvert t_2\rvert$ | 0,14–0,71 | **≈ 1,0** en todas las distancias |

En el cuadrado CCSD falla en toda la curva, incluso cerca del mínimo, porque el problema no es estirar enlaces sino la casi degeneración: las amplitudes dobles valen ~1 (la referencia HF no domina la función de onda). El barrido del rectángulo a 1,2 Å lo confirma: al alargar un lado ($a$ = 1 → 2) la brecha crece de 26 a 221 mHa, $\max\lvert t_2\rvert$ baja de 0,99 a 0,14 y el error de CCSD pasa de −6,5 a +0,01 mHa.

**Ansatz.** Se usó el elegido en la sección 3 (2-QUpCCGSD podado, 19 parámetros, 465 `cz`), aunque su poda se ajustó sobre la cadena lineal. **Sigue llegando a FCI en el cuadrado**: con 10 arranques el mejor queda a ≤ 0,10 mHa en todas las distancias (statevector, solo para elegir θ). Pero el paisaje es peor: solo 4–9 de 10 arranques llegan a precisión química (en el lineal, 3–9 de 10), mientras que el 2-QUpCCGSD completo (24 parámetros, 590 `cz`) llega con 7–10 de 10 y siempre a ≤ 1e-6 mHa con ⟨S²⟩ ~ 1e-14. El podado da ⟨S²⟩ ≤ 4e-4 en el cuadrado (algo de triplete, que a 3,0 Å está a 0,7 mHa). Conclusión: la poda no rompe la expresividad en el cuadrado, pero lo vuelve más dependiente del multi-start; si hiciera falta robustez, el completo cuesta 125 `cz` más.

**Medición (stand-in de #6).** `qwc_sampler_energy`: grupos QWC de `SparsePauliOp.group_commuting(qubit_wise=True)` (46 grupos en el lineal, 33–35 en el cuadrado), una rotación de base por grupo (X: H; Y: S†H), shots por grupo ∝ $\sum_k \lvert c_k\rvert$ del grupo, $3\times10^6$ shots por punto. $E$ y $\sigma$ salen de los bitstrings de `SamplerV2`: por cada bitstring se evalúa el grupo entero (así entran las covarianzas dentro del grupo), $\sigma^2 = \sum_g \text{Var}_g / N_g$. Validación: con 20 semillas a 1,8 Å (cuadrado, $2\times10^5$ shots) la dispersión es 4,7 mHa y la σ calculada 4,8 mHa; todas las energías medidas quedan a ≤ 2,4 σ del valor statevector de sus θ.

**Resultados** (energías medidas; tabla `geo_summary`):

| | lineal | cuadrado |
|---|---|---|
| $R_e$ FCI / VQE | 0,890 / 0,897 ± 0,008 Å | 1,279 / 1,285 ± 0,012 Å |
| $D_e$ FCI / VQE | 313,2 / 310,7 ± 1,9 mHa | 103,1 / 105,2 ± 1,6 mHa |
| $R_e$ CCSD | 0,890 Å | 1,304 Å |
| $D_e$ CCSD | 290,2 mHa (−23 mHa) | no definido (no converge a 3,0 Å) |
| σ por punto | 1,2–1,7 mHa | 1,1–1,3 mHa |
| máx. \|VQE − FCI\| | 3,4 mHa (2,6 σ) | 2,2 mHa (1,8 σ) |

El error del VQE medido es **ruido de shots**: el error statevector de los mismos θ es ≤ 0,1 mHa, y $\lvert E-E_\text{FCI}\rvert$ queda a ≤ 2,6 σ en todos los puntos. En el cuadrado el VQE le gana a CCSD ($\lvert E_\text{VQE}-E_\text{FCI}\rvert + \sigma < \lvert E_\text{CCSD}-E_\text{FCI}\rvert$) en todas las distancias desde 0,9 Å; en el lineal, solo desde ~1,6 Å, que es donde CCSD empieza a fallar.

**Qué cambia respecto de la cadena lineal y por qué.**
- **Curva más baja y ancha, con el mínimo más lejos** ($R_e$ 1,28 vs. 0,89 Å) y **$D_e$ tres veces menor** (103 vs. 313 mHa). En el cuadrado cada H tiene dos vecinos a $d$ y uno a $\sqrt2\,d$ (más repulsión nuclear), y los 4 electrones llenan un orbital enlazante y la mitad de un par degenerado no enlazante: es el análogo de H₄ del ciclobutadieno antiaromático. El cuadrado es en realidad un estado de transición entre dos rectángulos (dos H₂), como muestra el barrido del rectángulo. A 3,0 Å las dos formas llegan al mismo límite de 4 átomos (−1,867 Ha).
- **CCSD falla en toda la curva**, no solo en la zona estirada, porque la referencia es casi degenerada desde el principio (ver tabla). El ansatz de pares generalizados no depende de que HF domine, así que su error no cambia con la forma.
- **El paisaje del VQE es más difícil** (menos arranques buenos con el ansatz podado), y la brecha singlete–triplete es chica en las dos puntas de la curva (1,5 mHa a 0,8 Å, 0,7 mHa a 3,0 Å), así que hay que mirar ⟨S²⟩ además de la energía.

**Qué no funcionó o descartamos.**
- **Varios PUBs con distinta cantidad de shots en un mismo job de `SamplerV2`** (qiskit-aer 0.17.2): sus muestras salen **correlacionadas** (correlación 0,64 entre los conteos de dos circuitos de 1 qubit con 2000 y 3000 shots; con la misma cantidad de shots no pasa). Con un job por punto, los grupos quedaban correlacionados y la dispersión real era ~1,45 veces la σ calculada (10,6 vs. 7,3 mHa en 60 repeticiones). Se resolvió con **un job por grupo y semilla propia** (`SeedSequence([seed, g])`). **Ojo #6:** el reparto de shots por grupo dispara este problema.
- Reparto óptimo $N_g \propto \sqrt{\text{Var}_g}$ con un piloto de 1000 shots por grupo: lo probamos primero, pero la validación de σ fallaba (en parte por el problema anterior, y porque reusar la semilla del piloto en la corrida principal repite las mismas muestras). Nos quedamos con $\propto \sum\lvert c_k\rvert$, que no necesita piloto; con varianzas exactas, el óptimo ahorraría ~20 % de shots para la misma σ.
- PySCF con los hilos de OpenMP por defecto: con matrices de 4×4 CCSD tardaba ~10 veces más (66 s vs. 5 s para las referencias); se usa un hilo solo en esta celda (`with_omp_threads(1)`).
- `get_int_counts()` sobre 3 millones de shots es un loop de Python (~30 s por punto); se cuenta con `np.bincount` sobre el arreglo de bits.

**Limitaciones.**
- σ ≈ 1,2–1,7 mHa es del orden de la precisión química (1,6 mHa): la precisión química de los puntos medidos se cumple solo en promedio, no con margen. Bajarla a ~0,5 mHa costaría ~10 veces más shots con QWC; la agrupación por rotación de base (`docs/analysis/`) sería el camino.
- Los últimos dígitos dependen de qué celdas corrieron antes (BFGS cae en óptimos levemente distintos y cambian los θ medidos): entre corridas vimos moverse el error máximo medido del lineal entre 3,1 y 3,4 mHa.
- Grilla reducida: $R_e$ sale de un spline sobre 3–4 puntos cerca del mínimo (el de FCI sobre esa misma grilla es la referencia justa); la incertidumbre de $R_e$ y $D_e$ es un bootstrap paramétrico sobre σ.
- Solo se midió el ansatz podado; el completo se evaluó con statevector.
- Las semillas de la medición están fijas, pero BFGS puede caer en óptimos apenas distintos entre corridas (diferencias numéricas mínimas), así que las últimas cifras de la tabla pueden moverse unas décimas de mHa.
- Tiempo: ~3 min la subsección entera (casi todo es muestrear $6\times10^7$ shots).

### Comparación de ansätze (#15)

Comparamos cinco ansätze en **toda la grilla** (19 distancias). Todos arrancan en `HF_BITSTRING`:

| Ansatz | Qué es | Conserva |
|---|---|---|
| **HEA (RY + cz, 4 capas)** | Hardware-efficient tipo Kandala et al. (2017): capas de `ry` y una cadena de `cz` en línea | nada (ni N, ni S_z, ni S²) |
| **UCCSD** | Coupled cluster unitario, un paso de Trotter: dobles y después simples (occ → virt) | N, S_z |
| **k-UpCCGSD** (k = 1, 2, 3) | Lee et al. (2019): por capa, dobles de pares `(pα, pβ) → (qα, qβ)` y simples generalizadas con el mismo θ para α y β | N, S_z, S² |

**Cómo se mide cada cosa**

- **Parámetros (vale statevector):** simulador numpy propio con gradiente analítico (método adjunto) y L-BFGS-B. Los ansätze que conservan N y S_z se simulan en el sector de 36 estados en lugar de 256. Se barre la grilla con warm start en las dos direcciones, con 4 arranques en cada extremo (θ = 0 y 3 al azar), y se queda la menor energía por punto.
- **Control cruzado:** para cada punto, el circuito de Qiskit **transpilado** da la misma energía que el simulador numpy (`assert` a 1e-7 Ha).
- **`cz` y profundidad:** con `transpile_line` de la celda 0.1 (base `cz, sx, x, rz`, conectividad en línea, `optimization_level=3`). Las excitaciones fermiónicas se compilan como `PauliEvolutionGate` de sus cadenas de Jordan-Wigner.
- **Error medido:** `SamplerV2` de `qiskit_aer` con el **mismo presupuesto para todos**: 2²⁰ shots por punto, repartidos entre los grupos QWC en proporción a la norma 1 de sus coeficientes. E y σ salen de los bitstrings.
- **⟨S²⟩:** del estado final. Detecta contaminación con el triplete, que a 3,0 Å está a 0,46 mHa del fundamental.
- **Con ruido:** si #11 define `noise_model` (de `ruido.py`), se mide también con ruido en `d ≥ 1,8 Å` (matriz densidad, 2¹⁸ shots por punto).

El Hamiltoniano es `qubit_hamiltonian(d)` de la sección 2 (#2), y los operadores de escalera son los mismos (`_jw_ladder`).

**Resultados guardados.** Recalcular todo tarda ~8 min (~10 min más con ruido), así que por defecto la celda usa las tablas guardadas en `ANSATZ_CACHE` (de una corrida con estos mismos parámetros) y solo arma el resumen y los gráficos. Para recalcular: `RUN_ANSATZ_COMPARISON = True`.

In [ ]:
# Resultados guardados de la comparación de ansätze (#15). Recalcularlos tarda ~8 min, así que la celda
# siguiente los usa por defecto (RUN_ANSATZ_COMPARISON = False). Salieron de esa misma celda con True;
# para regenerarlos: poner True, correr y pegar acá results/ansatz_summary.csv y results/ansatz_comparison.csv.
ANSATZ_CACHE = {
    "origen": "corrida del 2026-11-21, 2²⁰ shots por punto, sin ruido",
    "summary": """\
ansatz,params,cz,depth,shots/punto,t (s)
"HEA (RY+cz, 4 capas)",40,28,35,1048575,70.8
UCCSD,26,1303,4220,1048575,62.9
1-UpCCGSD,12,474,1451,1048576,48.2
2-UpCCGSD,24,982,2915,1048576,103
3-UpCCGSD,36,1490,4379,1048576,156.8
""",
    "comparison": """\
ansatz,d,err_sv_mHa,err_meas_mHa,sigma_mHa,s2,nfev,err_noisy_mHa,sigma_noisy_mHa
"HEA (RY+cz, 4 capas)",0.6,28.1825,26.8938,2.63636,0.00156709,163,,
"HEA (RY+cz, 4 capas)",0.7,35.1411,29.0508,2.52857,0.00218068,104,,
"HEA (RY+cz, 4 capas)",0.8,43.3454,47.4434,2.45431,0.00288881,116,,
"HEA (RY+cz, 4 capas)",0.9,53.0817,52.3082,2.40897,0.00370608,222,,
"HEA (RY+cz, 4 capas)",1,64.7266,61.7675,2.37955,0.00466501,161,,
"HEA (RY+cz, 4 capas)",1.1,78.6893,79.3273,2.36247,0.00580673,305,,
"HEA (RY+cz, 4 capas)",1.2,93.6281,91.5721,2.41864,0.308266,3165,,
"HEA (RY+cz, 4 capas)",1.3,98.4228,92.647,2.46322,0.786907,125,,
"HEA (RY+cz, 4 capas)",1.4,93.1391,93.6263,2.44505,1.06858,85,,
"HEA (RY+cz, 4 capas)",1.5,82.2726,83.9773,2.40746,1.25899,96,,
"HEA (RY+cz, 4 capas)",1.6,69.2302,63.5915,2.36147,1.4117,62,,
"HEA (RY+cz, 4 capas)",1.7,56.1066,55.504,2.322,1.54195,56,,
"HEA (RY+cz, 4 capas)",1.8,44.1041,41.6511,2.28246,1.65174,49,,
"HEA (RY+cz, 4 capas)",2,25.4069,25.7798,2.23158,1.81061,59,,
"HEA (RY+cz, 4 capas)",2.2,13.7091,20.4713,2.20551,1.90254,22,,
"HEA (RY+cz, 4 capas)",2.4,7.08725,6.62226,2.20529,1.95156,21,,
"HEA (RY+cz, 4 capas)",2.6,3.54889,3.50976,2.21012,1.97657,20,,
"HEA (RY+cz, 4 capas)",2.8,1.72681,0.242384,2.2253,1.98897,20,,
"HEA (RY+cz, 4 capas)",3,0.81643,2.80267,2.23513,1.99494,19,,
UCCSD,0.6,0.00572607,4.41402,2.68983,4.37935e-19,14,,
UCCSD,0.7,0.00733863,-5.5938,2.6074,1.43506e-18,13,,
UCCSD,0.8,0.0152497,0.977777,2.56645,9.63256e-18,13,,
UCCSD,0.9,0.0371981,-7.45098,2.54101,5.41812e-18,13,,
UCCSD,1,0.0852724,4.5116,2.52809,1.16769e-16,17,,
UCCSD,1.1,0.176873,-4.25244,2.52149,2.84358e-16,14,,
UCCSD,1.2,0.334807,3.76587,2.52154,8.80198e-15,14,,
UCCSD,1.3,0.584819,3.70552,2.52206,9.90329e-14,14,,
UCCSD,1.4,0.9492,5.09126,2.51968,2.23873e-13,16,,
UCCSD,1.5,1.43658,2.62862,2.51658,5.80612e-14,18,,
UCCSD,1.6,2.02955,2.32065,2.50346,4.5476e-13,18,,
UCCSD,1.7,2.67152,2.81257,2.4871,1.67786e-12,20,,
UCCSD,1.8,3.25205,6.15402,2.46613,2.72276e-15,26,,
UCCSD,2,3.54357,3.2924,2.41163,3.99049e-13,35,,
UCCSD,2.2,2.29129,0.404783,2.39175,0.112206,28,,
UCCSD,2.4,1.10441,-1.69757,2.38475,0.113269,36,,
UCCSD,2.6,0.49051,3.14259,2.39358,0.0995992,45,,
UCCSD,2.8,0.208136,-0.915683,2.39633,0.0824785,44,,
UCCSD,3,0.0861095,3.52078,2.41919,0.0662304,3,,
1-UpCCGSD,0.6,10.1778,8.57839,2.6574,5.27969e-25,21,,
1-UpCCGSD,0.7,12.0536,8.75324,2.58352,2.45413e-35,10,,
1-UpCCGSD,0.8,13.0704,15.8611,2.53942,2.65713e-19,49,,
1-UpCCGSD,0.9,14.0548,16.2274,2.50001,1.07337e-34,43,,
1-UpCCGSD,1,14.9955,14.9839,2.48542,8.00467e-34,98,,
1-UpCCGSD,1.1,15.8454,19.208,2.47435,5.27653e-21,36,,
1-UpCCGSD,1.2,16.5146,17.0844,2.46904,1.31351e-18,29,,
1-UpCCGSD,1.3,16.88,16.4728,2.46684,1.56144e-18,31,,
1-UpCCGSD,1.4,16.8099,18.3218,2.46443,-3.73341e-18,29,,
1-UpCCGSD,1.5,16.2059,13.9196,2.4643,1.9533e-33,27,,
1-UpCCGSD,1.6,15.0475,18.4423,2.45384,-4.75324e-18,24,,
1-UpCCGSD,1.7,13.4175,13.2529,2.43883,-5.42634e-18,43,,
1-UpCCGSD,1.8,11.4874,14.7318,2.42672,-2.43706e-19,23,,
1-UpCCGSD,2,7.54312,6.70262,2.39407,3.16623e-19,26,,
1-UpCCGSD,2.2,4.41885,2.08338,2.37912,-6.24107e-18,26,,
1-UpCCGSD,2.4,2.39676,2.72364,2.37942,6.08774e-33,28,,
1-UpCCGSD,2.6,1.23267,5.36989,2.39224,4.79978e-33,34,,
1-UpCCGSD,2.8,0.608199,-2.46581,2.39659,1.0543e-32,24,,
1-UpCCGSD,3,0.289347,0.495346,2.41843,1.38387e-17,3,,
2-UpCCGSD,0.6,4.583e-10,1.41951,2.6837,2.95846e-19,3,,
2-UpCCGSD,0.7,4.44977e-10,-5.509,2.60393,-5.03209e-19,299,,
2-UpCCGSD,0.8,6.75016e-10,7.09576,2.55845,-3.04819e-19,336,,
2-UpCCGSD,0.9,4.57412e-10,-1.98067,2.53016,8.9572e-19,174,,
2-UpCCGSD,1,2.53131e-10,2.5444,2.51366,-6.74623e-19,145,,
2-UpCCGSD,1.1,3.29514e-10,-4.22836,2.50446,1.08604e-18,149,,
2-UpCCGSD,1.2,1.03917e-10,1.31878,2.49996,-1.5746e-18,153,,
2-UpCCGSD,1.3,1.56763e-10,0.0527445,2.49404,1.18316e-18,134,,
2-UpCCGSD,1.4,6.44818e-10,0.980463,2.48734,7.23949e-18,125,,
2-UpCCGSD,1.5,1.69198e-10,-1.38094,2.47477,-1.46723e-18,116,,
2-UpCCGSD,1.6,7.37188e-11,-1.32799,2.46238,-7.63897e-18,96,,
2-UpCCGSD,1.7,2.22933e-10,-5.16801,2.43934,3.50295e-18,108,,
2-UpCCGSD,1.8,6.30607e-11,1.62632,2.4186,-2.65691e-18,100,,
2-UpCCGSD,2,5.7554e-10,-1.0067,2.3677,-2.42194e-17,104,,
2-UpCCGSD,2.2,8.9706e-11,-3.65695,2.3337,5.13917e-32,120,,
2-UpCCGSD,2.4,3.4639e-11,-1.82842,2.32428,-1.53576e-17,135,,
2-UpCCGSD,2.6,4.44089e-11,-0.859931,2.32188,-1.98845e-17,125,,
2-UpCCGSD,2.8,9.41025e-10,-0.960527,2.32813,-1.125e-17,114,,
2-UpCCGSD,3,1.10134e-10,-1.15899,2.34193,5.33076e-32,142,,
3-UpCCGSD,0.6,2.35367e-10,-1.43662,2.68337,6.26632e-20,3,,
3-UpCCGSD,0.7,1.66978e-10,-0.33559,2.60912,-3.57219e-19,123,,
3-UpCCGSD,0.8,3.20632e-10,-2.32172,2.5593,-6.37748e-19,115,,
3-UpCCGSD,0.9,3.16192e-10,0.506899,2.52963,6.60693e-32,110,,
3-UpCCGSD,1,4.94715e-10,-4.94887,2.5111,1.09528e-18,97,,
3-UpCCGSD,1.1,4.05898e-10,-1.15193,2.50215,-1.32167e-18,111,,
3-UpCCGSD,1.2,4.64517e-10,-3.17039,2.49887,4.10515e-32,96,,
3-UpCCGSD,1.3,1.30562e-10,-0.970725,2.4921,6.21738e-18,89,,
3-UpCCGSD,1.4,9.23706e-11,-2.13826,2.48931,-2.95894e-18,98,,
3-UpCCGSD,1.5,1.40776e-10,-0.790646,2.47872,-4.86695e-18,77,,
3-UpCCGSD,1.6,2.75335e-11,-0.177602,2.46116,-2.20627e-18,83,,
3-UpCCGSD,1.7,1.77636e-12,-1.38698,2.44142,5.9522e-18,81,,
3-UpCCGSD,1.8,8.83738e-11,-4.34975,2.41389,-6.40586e-18,98,,
3-UpCCGSD,2,4.04121e-10,1.82549,2.37154,-1.41666e-17,83,,
3-UpCCGSD,2.2,1.11022e-11,1.9776,2.33846,1.06538e-17,96,,
3-UpCCGSD,2.4,9.10383e-11,0.15998,2.32505,1.0987e-17,81,,
3-UpCCGSD,2.6,9.32587e-12,1.54444,2.32754,7.58006e-32,75,,
3-UpCCGSD,2.8,1.36779e-10,3.78092,2.33411,1.12503e-17,81,,
3-UpCCGSD,3,1.28786e-11,-0.3995,2.33706,-4.26998e-18,82,,
""",
}

In [ ]:
# Extras · Comparación de ansätze (#15)
# Para cada familia y cada d de la grilla: error vs. FCI (statevector y medido con SamplerV2),
# cz y profundidad tras transpilar en línea, parámetros, evaluaciones del optimizador y <S²>.
# Si existe `noise_model` (#11, ruido.py), también el error medido con ruido.
RUN_ANSATZ_COMPARISON = False  # True: recalcular todo (~8 min). False: usar los resultados guardados en ANSATZ_CACHE
NOISY_SHOTS_PER_POINT = 2 ** 18
import io
import itertools
import time
from functools import reduce

import scipy.sparse as sp
from qiskit.circuit import Parameter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit_aer.primitives import SamplerV2
from scipy.optimize import minimize

# --- Operadores fermiónicos en Jordan-Wigner -------------------------------------
# Los de la sección 2 (#2): a_j = Z_0 ... Z_{j-1} (X_j + iY_j)/2, qubit p = (p, α), qubit 4 + p = (p, β).
_AD, _A = _jw_ladder(N_QUBITS)


def _s2_operator() -> SparsePauliOp:
    """S² = S_z² + S_z + S₋S₊ (en JW, con la convención de la celda 0.1)."""
    n = lambda q: _AD[q] @ _A[q]
    sz = 0.5 * SparsePauliOp.sum([n(alpha_qubit(p)) - n(beta_qubit(p)) for p in range(N_ORB)])
    s_plus = SparsePauliOp.sum([_AD[alpha_qubit(p)] @ _A[beta_qubit(p)] for p in range(N_ORB)])
    return (sz @ sz + sz + s_plus.adjoint() @ s_plus).simplify(atol=1e-12)


S2_OP = _s2_operator()

# --- Ansätze --------------------------------------------------------------------
# Cada ansatz es HF + una lista de factores exp(θ_k G_k), con G_k antihermítico y real.
#  - "exc": excitación fermiónica G = T − T†. Cumple G³ = −G => exp(θG) = I + sinθ G + (1 − cosθ) G².
#  - "rot": rotación de Pauli G = −iP/2   =>  exp(θG) = cos(θ/2) I + sin(θ/2) (−iP).
# Varios factores pueden compartir parámetro (k-UpCCGSD empareja α y β).


def _excitation(creates: list[int], annihilates: list[int]) -> SparsePauliOp:
    """Generador hermítico i(T − T†) con T = a†_c1 a†_c2 ... a_a2 a_a1."""
    T = reduce(lambda x, y: x @ y, [_AD[c] for c in creates] + [_A[a] for a in reversed(annihilates)])
    return (1j * (T - T.adjoint())).simplify(atol=1e-12)


def _single(p: int, q: int, s: int) -> SparsePauliOp:  # (q, s) -> (p, s)
    return _excitation([_so(p, s)], [_so(q, s)])


def _double(p: int, q: int, r: int, s_: int, s1: int, s2: int) -> SparsePauliOp:  # (r s1, s_ s2) -> (p s1, q s2)
    return _excitation([_so(p, s1), _so(q, s2)], [_so(r, s1), _so(s_, s2)])


OCC, VIRT = range(N_ALPHA), range(N_ALPHA, N_ORB)


def ansatz_uccsd():
    """UCCSD de un paso de Trotter (occ -> virt, conservan S_z). Orden: dobles y después simples."""
    ops = []
    for s in (0, 1):  # dobles con el mismo espín
        ops += [_double(a, b, i, j, s, s) for i, j in itertools.combinations(OCC, 2)
                for a, b in itertools.combinations(VIRT, 2)]
    ops += [_double(a, b, i, j, 0, 1) for i in OCC for j in OCC for a in VIRT for b in VIRT]  # αβ
    ops += [_single(a, i, s) for s in (0, 1) for i in OCC for a in VIRT]
    return [("exc", g, k) for k, g in enumerate(ops)], len(ops)


def ansatz_kupccgsd(k: int):
    """k-UpCCGSD (Lee et al. 2019): por capa, dobles de pares y después simples generalizadas
    (θ compartido entre α y β, así el ansatz conserva S²)."""
    gens, n = [], 0
    for _ in range(k):
        for p, q in itertools.combinations(range(N_ORB), 2):  # par (p α, p β) -> (q α, q β)
            gens.append(("exc", _double(q, q, p, p, 0, 1), n))
            n += 1
        for p, q in itertools.combinations(range(N_ORB), 2):
            gens += [("exc", _single(q, p, 0), n), ("exc", _single(q, p, 1), n)]
            n += 1
    return gens, n


def ansatz_hea(layers: int):
    """Hardware-efficient (Kandala et al. 2017, versión real): capas de RY + cadena de cz en línea."""
    gens, n = [], 0
    for layer in range(layers + 1):
        for q in range(N_QUBITS):
            gens.append(("rot", SparsePauliOp.from_sparse_list([("Y", [q], 1.0)], N_QUBITS), n))
            n += 1
        if layer < layers:
            gens.append(("cz", None, None))
    return gens, n


ANSATZ_FACTORIES = {
    "HEA (RY+cz, 4 capas)": lambda: ansatz_hea(4),
    "UCCSD": ansatz_uccsd,
    "1-UpCCGSD": lambda: ansatz_kupccgsd(1),
    "2-UpCCGSD": lambda: ansatz_kupccgsd(2),
    "3-UpCCGSD": lambda: ansatz_kupccgsd(3),
}

# --- Circuito de Qiskit equivalente (para contar cz y para medir) ---------------


def ansatz_circuit(gens, n_params: int) -> tuple[QuantumCircuit, list[Parameter]]:
    theta = [Parameter(f"θ{k:02d}") for k in range(n_params)]
    qc = QuantumCircuit(N_QUBITS)
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for kind, gen, k in gens:
        if kind == "cz":
            for q in range(N_QUBITS - 1):
                qc.cz(q, q + 1)
        elif kind == "rot":
            qc.ry(theta[k], int(np.flatnonzero(gen.paulis.x[0])[0]))  # exp(−iθY/2)
        else:  # exp(θ G) = exp(−i θ (iG)), y gen ya es iG (hermítico)
            qc.append(PauliEvolutionGate(gen, time=theta[k]), range(N_QUBITS))
    return qc, theta


# --- Simulador numpy con gradiente adjunto (solo para buscar parámetros) -------
_CZ_DIAG = np.array([(-1) ** sum((b >> q) & (b >> (q + 1)) & 1 for q in range(N_QUBITS - 1))
                     for b in range(2 ** N_QUBITS)], dtype=float)


def _real_generator(kind, gen):
    """Matriz real y rala del generador antihermítico G."""
    M = gen.to_matrix(sparse=True)  # = iG (exc) o P (rot)
    G = (-1j * M) if kind == "exc" else (-0.5j * M)
    assert abs(G.imag).max() < 1e-12
    return sp.csr_matrix(G.real)


# SECTOR (sección 2): los 36 estados con N = 4, S_z = 0. Acá solo se usa como conjunto de índices.


class NumpyAnsatz:
    """Simulador del ansatz. Si todos los factores conservan N y S_z, trabaja en el sector de 36 estados."""

    def __init__(self, gens, n_params):
        self.n = n_params
        self.idx = SECTOR if all(kind == "exc" for kind, _, _ in gens) else np.arange(2 ** N_QUBITS)
        self.ops = [(kind, None if kind == "cz" else self.proj(_real_generator(kind, gen)), k)
                    for kind, gen, k in gens]
        self.hf = (self.idx == int(HF_BITSTRING, 2)).astype(float)

    def proj(self, M):
        return sp.csr_matrix(M[self.idx][:, self.idx])

    @staticmethod
    def _apply(kind, G, t, v):
        if kind == "cz":  # solo en el espacio completo (HEA)
            return _CZ_DIAG * v
        Gv = G @ v
        if kind == "exc":
            return v + np.sin(t) * Gv + (1 - np.cos(t)) * (G @ Gv)
        return np.cos(t / 2) * v + 2 * np.sin(t / 2) * Gv

    def state(self, theta):
        v = self.hf
        for kind, G, k in self.ops:
            v = self._apply(kind, G, None if k is None else theta[k], v)
        return v

    def energy_and_grad(self, theta, H):
        psi = self.state(theta)
        lam = H @ psi
        e = psi @ lam
        grad = np.zeros(self.n)
        for kind, G, k in reversed(self.ops):  # dE/dθ = 2 <λ_k| G |ψ_k>
            if kind != "cz":
                grad[k] += 2 * lam @ (G @ psi)
            t = None if k is None else -theta[k]
            psi, lam = self._apply(kind, G, t, psi), self._apply(kind, G, t, lam)
        return e, grad


def _minimize(model, H, x0):
    res = minimize(model.energy_and_grad, x0, args=(H,), jac=True, method="L-BFGS-B",
                   options={"maxiter": 5000, "ftol": 1e-14, "gtol": 1e-9})
    return res.x, res.fun, res.nfev


def optimize_grid(model, Hs, n_random=3, seed=0):
    """Barrido con warm start en las dos direcciones + multi-start en los extremos; por punto, el mejor."""
    rng = np.random.default_rng(seed)
    best = {}
    Hs = {d: model.proj(H) for d, H in Hs.items()}

    def keep(d, x, e, nfev):
        if d not in best or e < best[d][1] - 1e-12:
            best[d] = (x, e, nfev)

    def starts():
        return [np.zeros(model.n)] + [rng.normal(0, 0.1, model.n) for _ in range(n_random)]

    for order in (list(D), list(D)[::-1]):
        x = min((_minimize(model, Hs[order[0]], x0) for x0 in starts()), key=lambda r: r[1])[0]
        for d in order:
            x, e, nfev = _minimize(model, Hs[d], x)
            keep(d, x, e, nfev)
    return best


# --- Medición con SamplerV2 (grupos QWC, shots ∝ peso del grupo) -----------------
SHOTS_PER_POINT = 2 ** 20  # mismo presupuesto por punto para todos los ansätze
# Ojo: con SamplerV2(seed=s), Aer repite semillas entre pubs y las mediciones de distintos grupos
# quedan correlacionadas (la σ calculada subestima el error real). Una semilla independiente por pub lo evita.
_seeds = np.random.default_rng(1234)


def _rotate_to_z(qc: QuantumCircuit, basis_label: str) -> QuantumCircuit:
    m = qc.copy()
    for q, b in enumerate(reversed(basis_label)):
        if b == "X":
            m.h(q)
        elif b == "Y":
            m.sdg(q)
            m.h(q)
    m.measure_all()
    return transpile(m, basis_gates=BASIS_GATES, optimization_level=0)


def measure_energies(tqc, theta, xs, ops_l, shots_per_point=SHOTS_PER_POINT, noise_model=None):
    """E y σ medidos para cada fila de xs (un juego de parámetros por d), sobre el circuito transpilado.

    ops_l: SparsePauliOp ya con el layout del circuito transpilado, uno por d (mismo orden que xs).
    """
    labels = sorted(set().union(*[set(op.paulis.to_labels()) for op in ops_l]) - {"I" * N_QUBITS})
    groups = SparsePauliOp(labels).group_commuting(qubit_wise=True)
    coeff = [dict(zip(op.paulis.to_labels(), op.coeffs.real)) for op in ops_l]
    weight = np.array([np.mean([sum(abs(c.get(l, 0.0)) for l in g.paulis.to_labels()) for c in coeff]) for g in groups])
    shots = np.maximum(100, np.round(shots_per_point * weight / weight.sum())).astype(int)

    pubs = []
    for g, n in zip(groups, shots):
        basis = "".join(max(col, key=lambda ch: ch != "I") for col in zip(*g.paulis.to_labels()))
        circ = _rotate_to_z(tqc, basis)
        pubs.append((circ, np.array([[x[theta.index(p)] for p in circ.parameters] for x in xs]), int(n)))
    options = {} if noise_model is None else {
        "backend_options": {"method": "density_matrix", "noise_model": noise_model}}
    results = [SamplerV2(seed=int(_seeds.integers(2 ** 31)), options=options).run([pub]).result()[0]
               for pub in pubs]

    bits = np.array([[(b >> q) & 1 for q in range(N_QUBITS)] for b in range(2 ** N_QUBITS)])
    E = np.array([c.get("I" * N_QUBITS, 0.0) for c in coeff])
    var = np.zeros(len(xs))
    for g, res in zip(groups, results):
        ls = g.paulis.to_labels()
        masks = np.array([[ch != "I" for ch in reversed(l)] for l in ls])          # (términos, qubits)
        signs = (-1.0) ** (bits @ masks.T.astype(int))                              # (256, términos)
        for j in range(len(xs)):
            v = signs @ np.array([coeff[j].get(l, 0.0) for l in ls])               # valor por shot, por bitstring
            cnt = res.data.meas.get_int_counts(loc=j)
            idx, n_b = np.fromiter(cnt.keys(), int), np.fromiter(cnt.values(), float)
            n = n_b.sum()
            mean = n_b @ v[idx] / n
            E[j] += mean
            var[j] += (n_b @ (v[idx] - mean) ** 2) / (n - 1) / n
    return E, np.sqrt(var), int(shots.sum())


# --- Corrida (o resultados guardados) ----------------------------------------
# Con RUN_ANSATZ_COMPARISON = False se leen las tablas guardadas en ANSATZ_CACHE (celda anterior), que salieron
# de esta misma celda con True. El resumen y los gráficos de abajo se calculan igual en los dos casos.
e_fci = refs.set_index("d").e_fci
if RUN_ANSATZ_COMPARISON:
    t0 = time.time()
    NOISE_MODEL = globals().get("noise_model")  # #11: el NoiseModel de ruido.py (o la función ruido.noise_model)
    if callable(NOISE_MODEL):
        NOISE_MODEL = NOISE_MODEL()
    H_OPS = {d: qubit_hamiltonian(d) for d in D}
    H_MATS = {d: sp.csr_matrix(op.to_matrix(sparse=True).real) for d, (op, _) in H_OPS.items()}
    S2_MAT = sp.csr_matrix(S2_OP.to_matrix(sparse=True).real)

    rows, ansatz_summary = [], []
    for name, factory in ANSATZ_FACTORIES.items():
        t = time.time()
        gens, n_params = factory()
        model = NumpyAnsatz(gens, n_params)
        best = optimize_grid(model, H_MATS)

        qc, theta = ansatz_circuit(gens, n_params)
        tqc = transpile_line(qc)
        ops_l = [H_OPS[d][0].apply_layout(tqc.layout) for d in D]
        xs = [best[d][0] for d in D]

        # Control cruzado: el circuito de Qiskit transpilado da la misma energía que el simulador numpy.
        for d, x, op_l in zip(D, xs, ops_l):
            e_qiskit = Statevector(tqc.assign_parameters(dict(zip(theta, x)))).expectation_value(op_l).real
            assert abs(e_qiskit - best[d][1]) < 1e-7, f"{name}: circuito ≠ modelo en d={d}"

        e_meas, sigma, shots = measure_energies(tqc, theta, xs, ops_l)
        noisy = {}
        if NOISE_MODEL is not None:  # con ruido: solo la zona estirada (matriz densidad, es más lento)
            js = [j for j, d in enumerate(D) if d >= STRETCHED]
            e_n, s_n, _ = measure_energies(tqc, theta, [xs[j] for j in js], [ops_l[j] for j in js],
                                           NOISY_SHOTS_PER_POINT, NOISE_MODEL)
            noisy = {D[j]: (e_n[i], s_n[i]) for i, j in enumerate(js)}
        for j, d in enumerate(D):
            x, e_sv, nfev = best[d]
            psi, s2_mat = model.state(x), model.proj(S2_MAT)
            e_nuc = H_OPS[d][1]
            rows.append({"ansatz": name, "d": d,
                         "err_sv_mHa": 1e3 * (e_sv + e_nuc - e_fci[d]),
                         "err_meas_mHa": 1e3 * (e_meas[j] + e_nuc - e_fci[d]),
                         "sigma_mHa": 1e3 * sigma[j], "s2": psi @ (s2_mat @ psi), "nfev": nfev,
                         "err_noisy_mHa": 1e3 * (noisy[d][0] + e_nuc - e_fci[d]) if d in noisy else np.nan,
                         "sigma_noisy_mHa": 1e3 * noisy[d][1] if d in noisy else np.nan})
        ops = tqc.count_ops()
        ansatz_summary.append({"ansatz": name, "params": n_params, "cz": ops.get("cz", 0), "depth": tqc.depth(),
                        "shots/punto": shots, "t (s)": round(time.time() - t, 1)})
        print(f"{name:22s} listo en {time.time() - t:5.1f} s")

    ansatz_cmp = pd.DataFrame(rows)
    ansatz_base = pd.DataFrame(ansatz_summary).set_index("ansatz")
    os.makedirs("results", exist_ok=True)
    ansatz_cmp.to_csv("results/ansatz_comparison.csv", index=False)
    ansatz_base.to_csv("results/ansatz_summary.csv")
    print(f"Total: {time.time() - t0:.0f} s")
else:
    ansatz_cmp = pd.read_csv(io.StringIO(ANSATZ_CACHE["comparison"]))
    ansatz_base = pd.read_csv(io.StringIO(ANSATZ_CACHE["summary"])).set_index("ansatz")
    print(f"Resultados guardados ({ANSATZ_CACHE['origen']}). Para recalcular: RUN_ANSATZ_COMPARISON = True.")

stretched = ansatz_cmp[ansatz_cmp.d >= STRETCHED]
ansatz_summary = ansatz_base.copy()
ansatz_summary["max err sv (d ≥ 1,8) mHa"] = stretched.groupby("ansatz").err_sv_mHa.max()
ansatz_summary["max err sv (toda) mHa"] = ansatz_cmp.groupby("ansatz").err_sv_mHa.max()
ansatz_summary["max |err med|+σ (d ≥ 1,8) mHa"] = (stretched.err_meas_mHa.abs() + stretched.sigma_mHa).groupby(stretched.ansatz).max()
if ansatz_cmp.err_noisy_mHa.notna().any():
    ansatz_summary["max |err ruido| (d ≥ 1,8) mHa"] = stretched.err_noisy_mHa.abs().groupby(stretched.ansatz).max()
ansatz_summary["max <S²>"] = ansatz_cmp.groupby("ansatz").s2.max()
ansatz_summary["nfev medio"] = ansatz_cmp.groupby("ansatz").nfev.mean().round(0)
ansatz_summary["le gana a CCSD"] = stretched.assign(
    ok=lambda df: df.err_meas_mHa.abs() + df.sigma_mHa < 1e3 * (refs.set_index("d").e_ccsd - e_fci).abs()[df.d].values
).groupby("ansatz").ok.all()
print(ansatz_summary.to_string(float_format=lambda v: f"{v:.3g}"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for name, df in ansatz_cmp.groupby("ansatz", sort=False):
    ax1.semilogy(df.d, df.err_sv_mHa.clip(lower=1e-6), "o-", ms=3, label=f"{name} ({ansatz_summary.loc[name, 'cz']} cz)")
    ax2.errorbar(df.d, df.err_meas_mHa, yerr=df.sigma_mHa, fmt="o-", ms=3, capsize=2, label=name)
ccsd_err = 1e3 * (refs.e_ccsd - refs.e_fci).abs()
ax1.semilogy(refs.d, ccsd_err, "k--", label="|CCSD − FCI|")
for ax in (ax1, ax2):
    ax.axhline(1e3 * CHEM_ACC, color="C2", ls=":", label="precisión química")
    ax.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0)
    ax.set_xlabel("d (Å)")
ax1.set(ylabel="E_VQE − E_FCI (mHa)", title="Error con statevector (piso en 1e-6 mHa = exacto)")
ax2.plot(refs.d, ccsd_err, "k--")
ax2.set(ylabel="E_medida − E_FCI (mHa)", title=f"Error medido con SamplerV2 ({SHOTS_PER_POINT:.0e} shots/punto)")
ax1.legend(fontsize=7)
plt.tight_layout()
plt.show()

#### Resultados

| Ansatz | Parámetros | `cz` | Profundidad | Máx. error sv, d ≥ 1,8 (mHa) | Máx. error sv, toda la curva (mHa) | Máx. \|error medido\| + σ, d ≥ 1,8 (mHa) | Máx. ⟨S²⟩ | ¿Le gana a CCSD? |
|---|---:|---:|---:|---:|---:|---:|---:|:---:|
| HEA (4 capas) | 40 | 28 | 35 | 44,1 | 98,4 | 44–48 | 1,99 | no |
| UCCSD | 26 | 1303 | 4220 | 3,5 | 3,5 | 8,6 | 0,1 | sí (justo) |
| 1-UpCCGSD | 12 | 474 | 1451 | 11,5 | 16,9 | 17,2 | 0 | no |
| **2-UpCCGSD** | 24 | 982 | 2915 | **< 1e-9** | **< 1e-9** | 6,0 | 0 | **sí** |
| 3-UpCCGSD | 36 | 1490 | 4379 | < 1e-9 | < 1e-9 | 6,8 | 0 | sí |

Los números del HEA varían un poco entre corridas: su optimización tiene muchos mínimos locales y el álgebra lineal multihilo no es determinista.

#### Conclusión

- **2-UpCCGSD** es el que conviene si no hay ruido. Llega a FCI en toda la curva con solo 24 parámetros y conserva S² por construcción: no se contamina con el triplete casi degenerado de la zona estirada. 3-UpCCGSD no agrega nada y suma 50 % más de `cz`.
- **UCCSD** se traba en 3,5 mHa entre 1,8 y 2,2 Å. Le gana a CCSD por poco a 1,8 Å (8,6 contra 9,1 mHa) y nunca llega a precisión química en esa zona. **1-UpCCGSD** no alcanza.
- El **HEA** es 35 veces más barato en `cz`, pero se queda en mínimos locales (44 mHa a 1,8 Å) y rompe la simetría de espín (⟨S²⟩ ≈ 2): energía baja no significa estado correcto.
- **Lo que limita al ansatz exacto es la medición**, no el ansatz. Con 2²⁰ shots por punto, σ ≈ 2,3 mHa. Para tener precisión química medida hacen falta ~8 veces más shots (σ ≈ 0,8 mHa) o una medición más eficiente (#6).
- **Con ruido, la elección se invierte.** En una prueba con un modelo sintético igual al del enunciado (depolarizante, 0,1 % por `cz` y 0,005 % por `sx`/`x`), 2-UpCCGSD queda a ~450 mHa de FCI y el HEA a 21–67 mHa. Con ~1000 `cz`, el error por compuerta lo domina todo. Reducir `cz` (excitaciones compiladas como compuertas de Givens o QEB en lugar de cadenas de Pauli, #4) es lo que hace falta para que el ansatz exacto sirva con ruido.

#### Qué no funcionó

- **El orden de los factores importa mucho.** Con simples antes que dobles, 1-UpCCGSD llega a 220 mHa de error a 3,0 Å (con el orden inverso, a 0,3 mHa). UCCSD llega a 5,2 mHa y se contamina con el triplete (⟨S²⟩ hasta 1,9). Por eso todos aplican primero las dobles y después las simples, que actúan como una rotación orbital final.
- **Más capas de HEA no ayudan.** Con 8 capas (72 parámetros) da exactamente el mismo error que con 4 y tarda el doble: es un problema de optimización (mínimos locales), no de expresividad. Lo sacamos de la comparación.
- **`SamplerV2(seed=...)` de Aer repite semillas entre pubs.** Las mediciones de distintos grupos quedaban correlacionadas, la σ subestimaba el error real y aparecían desvíos de 3–4σ. Con una semilla independiente por pub, los z-scores de los ansätze exactos tienen desvío 1,06 en 40 puntos (máx. |z| = 2,8): la σ está bien calibrada. **Ojo #6:** les pasa lo mismo si usan una semilla fija.